# D-23: Notebook 3 — Multi-Layer Enrichment (GO/NO-GO Decision Point)

## Research Question

**Does enriching text chunks with all 8 hierarchical context layers significantly improve retrieval quality beyond single-layer enrichment?**

This notebook executes the critical decision-point experiment for the FractalRecall project. Results determine whether we proceed with multi-layer enrichment infrastructure (D-24+) or simplify to single-layer enrichment + metadata filtering.

## Experimental Design: 3-Way Comparison

| Notebook | Approach | Enrichment | Key Question |
|----------|----------|-----------|--------------|
| **D-21** | Baseline | None (raw chunks only) | What is baseline retrieval performance? |
| **D-22** | Single-Layer | Document-level prefix (~15-25 tokens) | Does basic structural context help? |
| **D-23** | Multi-Layer | All 8 layers (~50-150 tokens per chunk) | Does comprehensive hierarchy justify token cost? |

All three notebooks use:
- Same 36-query ground-truth set (Q-01 through Q-36)
- Same corpus (Aethelgard Worldbuilding Corpus v5.0, 25 documents)
- Same hybrid chunking engine (adjusted chunk sizes for token budgets)
- Same evaluation metrics (Precision@5, Recall@10, NDCG@10, MRR)
- Same statistical methodology (Wilcoxon signed-rank with Bonferroni correction)

## The 8 Context Layers

| # | Layer | Purpose | Example | Est. Tokens |
|---|-------|---------|---------|-------------|
| 1 | **Corpus** | Dataset identity | "Corpus: Aethelgard Worldbuilding Corpus v5.0" | ~5 |
| 2 | **Domain** | Content category | "Domain: This content is from a faction document in the organizations category." | ~8 |
| 3 | **Entity** | Named entity described | "Entity: This content describes The Iron Covenant." | ~8 |
| 4 | **Authority** | Canon/editorial status | "Authority: This content is canonical and authoritative." | ~8 |
| 5 | **Temporal** | Time period | "Temporal: The events described span the Third Age and Fourth Age." | ~12 |
| 6 | **Relational** | Links to other entities | "Relationships: founded by Elena Voss; rival of Silver Hand." | ~15-30 |
| 7 | **Section** | Document heading | "Section: This content is from the Origins section." | ~10 |
| 8 | **Content** | The actual chunk text | *(the raw text)* | variable |

**Total enrichment prefix per chunk: ~50-150 tokens** (vs. ~15-25 for D-22, ~0 for D-21)

## Critical Context

> This notebook is the critical experiment. Results determine whether FractalRecall proceeds with multi-layer enrichment (D-24+) or simplifies to single-layer + metadata filtering.

## References

- **D-32**: COLAB-SESSION-CONTEXT.md (8-layer enrichment template, session context)
- **R-01**: Embedding Model Evaluation (expanded from v2-moe only to 3 models after 512-token discovery)
- **R-02**: Chunking Strategy Analysis (token budget allocation)
- **R-03**: Anthropic Contextual Retrieval (contextual prefix prepending technique)

## Note: R-01 Scope Expansion

R-01 was originally scoped to v2-moe performance only. After discovering the 512-token context window limitation (vs. the documented 8,192), scope expanded to include v1.5 and BGE-M3. This D-23 notebook runs against SELECTED_MODEL (defaulting to v1.5 after D-21 execution).

## ⚠️ Round 3 Fixes Applied

This notebook has been patched for Round 3 (re-run after Round 2 analysis).

**Fixes applied:**

1. **NDCG Bug** (Cell 14): Deduplicate retrieved IDs before DCG computation. Round 2 produced NDCG values up to 2.79 because multiple chunks from the same document each accumulated relevance gains, but the IDCG denominator only counted unique documents.

2. **Missing Layers** (Cells 5, 9): Fixed FIELD_MAP to include `temporal_markers` and `cross_references` from the corpus YAML frontmatter. Updated layer builders to handle the actual data formats. Previously, Temporal, Relational, and Chunk Sequence layers returned None for all documents.

3. **Chunking Parameters** (Cells 3, 8): Changed `max_chunk_tokens` from 600 → 1024 and `overlap` from 50 → 150 to match D-21/D-22. Reduced `prefix_reserve_tokens` from 150 → 100 (actual max overhead is 84 tokens). This isolates the enrichment effect from chunking geometry changes.

*Patch applied: 2026-02-16 21:37 by fix_d23_round3.py*


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 02: Install Dependencies

Installs all required packages for D-23 notebook execution.
Same package set as D-21 and D-22 for consistency.

Packages:
  - chromadb: Vector database for embedding storage and retrieval
  - sentence-transformers: Embedding model framework (Nomic v2-moe, v1.5)
  - FlagEmbedding: Embedding model framework (BAAI BGE-M3)
  - transformers: Hugging Face model infrastructure
  - torch: PyTorch backend for embedding computation
  - numpy, pandas: Numerical/data analysis
  - scipy: Statistical testing (Wilcoxon signed-rank)
  - scikit-learn: ML utilities
  - matplotlib, seaborn: Visualization
  - tqdm: Progress bars
  - pyyaml: YAML frontmatter parsing
  - umap-learn: Dimensionality reduction for embedding visualization
"""

# Install required packages (suppress verbose output with -q)
!pip install -q chromadb sentence-transformers FlagEmbedding transformers torch numpy pandas scipy scikit-learn matplotlib seaborn tqdm pyyaml umap-learn

# ============================================================================
# VERSION VERIFICATION
# ============================================================================

print("✓ Dependency Installation Complete\n")
print("Package Version Check:")
print("-" * 50)

packages_to_check = [
    'chromadb',
    'sentence_transformers',
    'torch',
    'numpy',
    'pandas',
    'scipy',
    'sklearn',
    'matplotlib',
    'seaborn',
    'tqdm',
    'yaml',
]

for package in packages_to_check:
    try:
        mod = __import__(package)
        version = getattr(mod, '__version__', 'installed (no version attr)')
        print(f"  {package:25} {version}")
    except ImportError:
        print(f"  {package:25} [IMPORT FAILED]")

print("-" * 50)
print("\n✓ All critical dependencies installed and verified.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 03: Imports, Configuration, Model Selection

Defines all imports, model configurations, and path constants for D-23.
Extends D-22 config with multi-layer enrichment parameters.

Key D-23 additions vs D-22:
  - prefix_reserve_tokens increased to 100-150 (from D-22's ~30) for 8-layer prefix
  - BONFERRONI_PAIRS = 3 for 3-way statistical correction
  - ADJUSTED_ALPHA = 0.05 / 3 ≈ 0.0167
  - CORPUS_LABEL constant for Corpus layer
  - Output paths for D-23 specific artifacts
"""

# ============================================================================
# STANDARD LIBRARY IMPORTS
# ============================================================================
import os
import sys
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional
import yaml
import re
import json
from datetime import datetime

# ============================================================================
# THIRD-PARTY IMPORTS
# ============================================================================
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings

import chromadb
from chromadb.config import Settings

warnings.filterwarnings('ignore')

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

# ── USER MUST UPDATE THIS AFTER RUNNING D-21 ──
# Default: "v1.5" — the expected winner based on 8192-token context window
# and strong general-purpose performance. Update to match D-21's actual winner.
SELECTED_MODEL = "v1.5"  # Options: "v2-moe", "v1.5", "bge-m3"

@dataclass
class ModelConfig:
    """Configuration for an embedding model.

    Attributes:
        name: Short identifier (e.g., "v1.5")
        hf_model_id: Hugging Face model path
        max_tokens: Model's maximum context window in tokens
        max_chunk_tokens: Target maximum chunk size (content + prefix)
        prefix_reserve_tokens: Tokens reserved for enrichment prefix
        embedding_dim: Output embedding dimensionality
        task_prefix_doc: Prefix for document indexing (empty for BGE-M3)
        task_prefix_query: Prefix for query encoding (empty for BGE-M3)
    """
    name: str
    hf_model_id: str
    max_tokens: int
    max_chunk_tokens: int
    prefix_reserve_tokens: int
    embedding_dim: int
    task_prefix_doc: str
    task_prefix_query: str

# Three models supported — same as D-21/D-22
MODELS: Dict[str, ModelConfig] = {
    "v2-moe": ModelConfig(
        name="v2-moe",
        hf_model_id="nomic-ai/nomic-embed-text-v2-moe",
        max_tokens=512,
        max_chunk_tokens=350,
        prefix_reserve_tokens=100,   # 8-layer prefix: ~80-150 tokens
        embedding_dim=768,
        task_prefix_doc="search_document: ",
        task_prefix_query="search_query: ",
    ),
    "v1.5": ModelConfig(
        name="v1.5",
        hf_model_id="nomic-ai/nomic-embed-text-v1.5",
        max_tokens=8192,
        max_chunk_tokens=1024,       # [FIX Round 3] Match D-21/D-22 for controlled comparison
        prefix_reserve_tokens=100,   # [FIX Round 3] Actual max overhead is 84 tokens; 100 is safe
        embedding_dim=768,
        task_prefix_doc="search_document: ",
        task_prefix_query="search_query: ",
    ),
    "bge-m3": ModelConfig(
        name="bge-m3",
        hf_model_id="BAAI/bge-m3",
        max_tokens=8192,
        max_chunk_tokens=1024,
        prefix_reserve_tokens=150,   # 8-layer prefix: ~80-150 tokens
        embedding_dim=1024,
        task_prefix_doc="",          # BGE-M3 doesn't use task prefixes
        task_prefix_query="",
    ),
}

# Validate selection
if SELECTED_MODEL not in MODELS:
    raise ValueError(
        f"SELECTED_MODEL '{SELECTED_MODEL}' not in {list(MODELS.keys())}. "
        f"Update after running D-21."
    )

MODEL_CONFIG = MODELS[SELECTED_MODEL]

# ============================================================================
# PATH CONSTANTS
# ============================================================================

# Corpus directory (D-20 output)
CORPUS_DIR = Path("./corpus")

# D-23 output directory
OUTPUT_DIR = Path("./d23-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ChromaDB persistence directory
CHROMADB_DIR = OUTPUT_DIR / "chromadb"
CHROMADB_DIR.mkdir(parents=True, exist_ok=True)

# Prior notebook results for 3-way comparison
D21_RESULTS_PATH = Path("./d21-output/d21_results.csv")
D22_RESULTS_PATH = Path("./d22-output/d22_results.csv")

# ============================================================================
# EVALUATION CONSTANTS
# ============================================================================

K_PRECISION = 5      # Precision@5
K_RECALL = 10        # Recall@10
K_NDCG = 10          # NDCG@10

# Metric column names used throughout the notebook
METRICS = ["precision@5", "recall@10", "ndcg@10", "mrr"]
METRIC_LABELS = ["P@5", "R@10", "NDCG@10", "MRR"]

# ============================================================================
# STATISTICAL TESTING CONSTANTS
# ============================================================================

SIGNIFICANCE_LEVEL = 0.05

# 3-way comparison requires Bonferroni correction:
# 3 pairwise comparisons: D-23 vs D-21, D-23 vs D-22, D-22 vs D-21
BONFERRONI_PAIRS = 3
ADJUSTED_ALPHA = SIGNIFICANCE_LEVEL / BONFERRONI_PAIRS  # ≈ 0.0167

# ============================================================================
# MULTI-LAYER ENRICHMENT CONSTANTS
# ============================================================================

# Corpus layer label (hardcoded for this corpus)
CORPUS_LABEL = "Aethelgard Worldbuilding Corpus v5.0"

# ============================================================================
# PRINT CONFIGURATION SUMMARY
# ============================================================================

print("✓ Imports and Configuration Complete\n")
print("=" * 70)
print("D-23 CONFIGURATION SUMMARY")
print("=" * 70)

print(f"\nModel Selection:")
print(f"  SELECTED_MODEL:       {SELECTED_MODEL}")
print(f"  Model ID:             {MODEL_CONFIG.hf_model_id}")
print(f"  Max tokens:           {MODEL_CONFIG.max_tokens}")
print(f"  Max chunk tokens:     {MODEL_CONFIG.max_chunk_tokens}")
print(f"  Prefix reserve:       {MODEL_CONFIG.prefix_reserve_tokens} tokens (multi-layer)")
print(f"  Effective content:    {MODEL_CONFIG.max_chunk_tokens - MODEL_CONFIG.prefix_reserve_tokens} tokens")
print(f"  Embedding dimension:  {MODEL_CONFIG.embedding_dim}")

print(f"\nEvaluation:")
print(f"  Metrics:              {', '.join(METRICS)}")
print(f"  Alpha (unadjusted):   {SIGNIFICANCE_LEVEL}")
print(f"  Bonferroni pairs:     {BONFERRONI_PAIRS}")
print(f"  Adjusted alpha:       {ADJUSTED_ALPHA:.4f}")

print(f"\nPaths:")
print(f"  Corpus:               {CORPUS_DIR}")
print(f"  Output:               {OUTPUT_DIR}")
print(f"  ChromaDB:             {CHROMADB_DIR}")
print(f"  D-21 results:         {D21_RESULTS_PATH}")
print(f"  D-22 results:         {D22_RESULTS_PATH}")

print(f"\nEnrichment:")
print(f"  Corpus label:         {CORPUS_LABEL}")
print(f"  Layers:               8 (Corpus, Domain, Entity, Authority, Temporal, Relational, Section, Content)")

print("=" * 70)
print("\n✓ Configuration complete. Ready for corpus loading.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 04: Ground-Truth Query Set (Aligned with D-21/D-22)

Defines 36 ground-truth queries for evaluation, organized by type:
  - SINGLE_HOP (11): Direct attribute lookups
  - MULTI_HOP (8): Cross-entity relationship traversal
  - AUTHORITY (5): Canonical vs. draft vs. superseded status questions
  - TEMPORAL (6): Time-based historical queries
  - EXPLORATORY (6): Open-ended relationship discovery

Same query set as D-21 Cell 05 and D-22 Cell 04 for consistent 3-way comparison.

Each query includes:
  - query_id: Unique identifier (e.g., "Q-01")
  - type: Category (SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY)
  - text: Natural language query string
  - relevant_docs: List of corpus files expected to be relevant
"""

@dataclass
class GroundTruthQuery:
    """A single ground-truth query with expected relevant documents.

    Attributes:
        query_id: Unique query identifier (e.g., "Q-01")
        query_text: Natural language query
        query_type: One of SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY
        expected_filenames: List of corpus filenames expected to be relevant
        relevance_scores: Dict mapping filename → relevance grade (1-3)
    """
    query_id: str
    query_text: str
    query_type: str
    expected_filenames: List[str]
    relevance_scores: Dict[str, int]

# ============================================================================
# FULL QUERY SET: 36 queries (identical to D-21/D-22)
# ============================================================================

# SINGLE_HOP QUERIES (11 total)
# These test direct lookups of facts, entities, and concepts

QUERIES = [
    {
        "query_id": "Q-01",
        "type": "SINGLE_HOP",
        "text": "What is the Echo-Cant communication system?",
        "relevant_docs": ["000-codex_echo-cant.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-02",
        "type": "SINGLE_HOP",
        "text": "What are the defining characteristics of the Void-Marked?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-03",
        "type": "SINGLE_HOP",
        "text": "Describe the Harrow-Sick condition and its effects.",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_medical-phenomena-entry.md"],
    },
    {
        "query_id": "Q-04",
        "type": "SINGLE_HOP",
        "text": "What is the ODIN Protocol?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-05",
        "type": "SINGLE_HOP",
        "text": "What is the \u00c6ther-Weave operating system?",
        "relevant_docs": ["standalone_aether-weave-os.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-06",
        "type": "SINGLE_HOP",
        "text": "Who are the Scavenger-Barons and what do they do?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-07",
        "type": "SINGLE_HOP",
        "text": "What is the Nine-Tiers architectural framework?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-27",
        "type": "SINGLE_HOP",
        "text": "What are the primary functions of the Warden-Host?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-28",
        "type": "SINGLE_HOP",
        "text": "Define the Glitch in the context of Aethelgard's history.",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-30",
        "type": "SINGLE_HOP",
        "text": "What is the Spell-Lock system?",
        "relevant_docs": ["000-codex_spell-lock.md", "standalone_aether-weave-os.md"],
    },
    {
        "query_id": "Q-34",
        "type": "SINGLE_HOP",
        "text": "Describe the Weir-Bone material and its properties.",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "000-resources_comprehensive-glossary.md"],
    },

    # MULTI_HOP QUERIES (8 total)
    # These require traversing relationships between multiple entities/concepts

    {
        "query_id": "Q-08",
        "type": "MULTI_HOP",
        "text": "How do Iron-Bane and God-Sleeper theological positions on Svin-fylking differ?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db02-wb_svin-fylking-assembled-entry.md"],
    },
    {
        "query_id": "Q-09",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Harrow-Sick and the Void-Marked?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-10",
        "type": "MULTI_HOP",
        "text": "How do the Scavenger-Barons use Spell-Lock in their operations?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "000-codex_spell-lock.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-11",
        "type": "MULTI_HOP",
        "text": "Explain the conflict between Warden-Host sanctuary protocols and Iron-Bane territorial claims.",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-12",
        "type": "MULTI_HOP",
        "text": "How does the ODIN Protocol interact with the \u00c6ther-Weave OS?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-29",
        "type": "MULTI_HOP",
        "text": "What role does the Echo-Cant system play in maintaining Warden-Host operations?",
        "relevant_docs": ["000-codex_echo-cant.md", "db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-31",
        "type": "MULTI_HOP",
        "text": "How do Void-Marked and Weir-Bone materials interact in salvage contexts?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db02-wb_weir-bone-assembled-entry.md", "db03-dc_salvage-operations-manual.md"],
    },
    {
        "query_id": "Q-35",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Nine-Tiers architecture and Svin-fylking religious doctrine?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },

    # AUTHORITY QUERIES (5 total)
    # These test knowledge of canonical vs. draft vs. superseded information

    {
        "query_id": "Q-13",
        "type": "AUTHORITY",
        "text": "What is the canonical explanation for how the Glitch occurred?",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-14",
        "type": "AUTHORITY",
        "text": "What are the established facts about the God-Sleeper movement origins?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-15",
        "type": "AUTHORITY",
        "text": "Which interpretations of Harrow-Sick etiology are considered canon?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-16",
        "type": "AUTHORITY",
        "text": "What is the official Warden-Host stance on Void-Marked rights?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-17",
        "type": "AUTHORITY",
        "text": "According to published sources, what materials constitute a valid Spell-Lock?",
        "relevant_docs": ["000-codex_spell-lock.md", "000-resources_comprehensive-glossary.md"],
    },

    # TEMPORAL QUERIES (6 total)
    # These test time-based historical retrieval across the Aethelgard timeline

    {
        "query_id": "Q-18",
        "type": "TEMPORAL",
        "text": "What major events happened in the first century after the Glitch (Year 0-100 PG)?",
        "relevant_docs": ["db03-dc_jotun-reader-chronology.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-19",
        "type": "TEMPORAL",
        "text": "When was the Warden-Host sanctuary established and what precipitated it?",
        "relevant_docs": ["db03-dc_sanctuary-establishment-record.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-20",
        "type": "TEMPORAL",
        "text": "Trace the chronological development of Iron-Bane theological doctrine.",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-21",
        "type": "TEMPORAL",
        "text": "What is the timeline of major salvage discoveries in the Aethelgard region?",
        "relevant_docs": ["db03-dc_salvage-operations-manual.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-22",
        "type": "TEMPORAL",
        "text": "When did the God-Sleeper movement gain significant political influence?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-23",
        "type": "TEMPORAL",
        "text": "Describe the sequence of events in the Scavenger-Baron contract dispute.",
        "relevant_docs": ["db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },

    # EXPLORATORY QUERIES (6 total)
    # These test open-ended discovery of related concepts and themes

    {
        "query_id": "Q-24",
        "type": "EXPLORATORY",
        "text": "What are the major political tensions in post-Glitch Aethelgard?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },
    {
        "query_id": "Q-25",
        "type": "EXPLORATORY",
        "text": "How do different factions view the technological salvage efforts?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_salvage-operations-manual.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },
    {
        "query_id": "Q-26",
        "type": "EXPLORATORY",
        "text": "What medical and physiological mysteries remain unsolved in Aethelgard?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md", "db02-wb_void-marked-assembled-entry.md"],
    },
    {
        "query_id": "Q-32",
        "type": "EXPLORATORY",
        "text": "What are the intersections between religious doctrine and technological systems in Aethelgard?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "000-codex_odin-protocol.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-33",
        "type": "EXPLORATORY",
        "text": "How do material properties (Weir-Bone, Void-Marked) influence cultural practices?",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-36",
        "type": "EXPLORATORY",
        "text": "What gaps exist in the documented understanding of Aethelgard's pre-Glitch history?",
        "relevant_docs": ["db03-dc_historical-theological-survey.md", "db03-dc_jotun-reader-chronology.md", "000-resources_comprehensive-glossary.md"],
    },
]

# ============================================================================
# BUILD GROUND_TRUTH_QUERIES from QUERIES (for compatibility with Cell 14)
# ============================================================================

GROUND_TRUTH_QUERIES: List[GroundTruthQuery] = []
for q in QUERIES:
    # Build uniform relevance scores (all docs score 2 by default)
    relevance = {doc: 2 for doc in q["relevant_docs"]}
    GROUND_TRUTH_QUERIES.append(GroundTruthQuery(
        query_id=q["query_id"],
        query_text=q["text"],
        query_type=q["type"],
        expected_filenames=q["relevant_docs"],
        relevance_scores=relevance,
    ))

# ============================================================================
# VALIDATION
# ============================================================================

query_type_counts = {}
for q in QUERIES:
    query_type_counts[q["type"]] = query_type_counts.get(q["type"], 0) + 1

assert len(QUERIES) == 36, f"Expected 36 queries, got {len(QUERIES)}"
assert query_type_counts.get("SINGLE_HOP", 0) == 11, f"Expected 11 SINGLE_HOP"
assert query_type_counts.get("MULTI_HOP", 0) == 8, f"Expected 8 MULTI_HOP"
assert query_type_counts.get("AUTHORITY", 0) == 5, f"Expected 5 AUTHORITY"
assert query_type_counts.get("TEMPORAL", 0) == 6, f"Expected 6 TEMPORAL"
assert query_type_counts.get("EXPLORATORY", 0) == 6, f"Expected 6 EXPLORATORY"

print(f"\u2713 All 36 ground-truth queries validated (aligned with D-21/D-22)")
print(f"  Distribution: {query_type_counts}")
print(f"  GROUND_TRUTH_QUERIES: {len(GROUND_TRUTH_QUERIES)} GroundTruthQuery objects")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 05: Field Mapping & Normalization

Normalizes raw YAML frontmatter fields to consistent canonical forms.
Handles variations in field names (e.g., "canon" vs "canonical_status")
and value formats (e.g., True vs "yes" vs "canonical").

Identical to D-21 Cell 05 and D-22 Cell 05 for consistency.

Functions:
  - normalize_canon_status(value) → str ("true"/"false")
  - normalize_authority_layer(value) → str ("primary"/"secondary"/"tertiary"/"unknown")
  - normalize_entity_type(value) → str ("character"/"location"/"faction"/etc.)
  - normalize_version(value) → str (semantic version like "1.0")
  - map_frontmatter(raw) → Dict[str, Any] (fully normalized metadata dict)
"""

# ============================================================================
# FIELD MAP: maps canonical key → list of possible raw frontmatter key names
# ============================================================================

FIELD_MAP: Dict[str, List[str]] = {
    "type":              ["type", "entity_type", "document_type", "doc_type", "category"],
    "name":              ["name", "entity_name", "title", "subject"],
    "canon":             ["canon", "canon_status", "canonical", "is_canonical"],
    "authority_layer":   ["authority_layer", "authority", "authority_level"],
    "era":               ["era", "age", "time_period", "historical_era", "eras", "temporal_markers"],
    "domain_layer":      ["domain_layer", "domain", "lore_category"],
    "related_entities":  ["related_entities", "relationships", "relations", "links", "see_also", "cross_references"],
    "version":           ["version", "doc_version", "revision"],
    "tags":              ["tags", "keywords", "labels"],
    "description":       ["description", "summary", "blurb"],
}


def normalize_canon_status(value: Any) -> str:
    """Normalize canon/canonical status to 'true' or 'false'.

    Handles: True/False booleans, 'yes'/'no', 'canonical'/'apocryphal',
    numeric 1/0, and string variations.

    Args:
        value: Raw canon value from frontmatter

    Returns:
        Normalized string: 'true' or 'false'
    """
    if value is None:
        return "false"
    if isinstance(value, bool):
        return "true" if value else "false"
    s = str(value).lower().strip()
    if s in ("true", "yes", "1", "canonical", "canon"):
        return "true"
    return "false"


def normalize_authority_layer(value: Any) -> str:
    """Normalize authority layer to canonical form.

    Maps various authority designations to: 'primary', 'secondary',
    'tertiary', or 'unknown'.

    Args:
        value: Raw authority value

    Returns:
        One of: 'primary', 'secondary', 'tertiary', 'unknown'
    """
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    if s in ("primary", "official", "canonical", "1"):
        return "primary"
    if s in ("secondary", "semi-official", "semi-canon", "2"):
        return "secondary"
    if s in ("tertiary", "unofficial", "fan", "apocryphal", "3"):
        return "tertiary"
    return "unknown"


def normalize_entity_type(value: Any) -> str:
    """Normalize entity type to canonical form.

    Maps variations like 'char', 'npc', 'org' to standard names.

    Args:
        value: Raw entity type value

    Returns:
        Normalized type string (e.g., 'character', 'faction', 'location')
    """
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    type_map = {
        "character": "character", "char": "character", "npc": "character", "person": "character",
        "faction": "faction", "org": "faction", "organization": "faction", "group": "faction",
        "location": "location", "place": "location", "region": "location", "area": "location",
        "event": "event", "battle": "event", "war": "event",
        "item": "item", "artifact": "item", "object": "item", "weapon": "item",
        "concept": "concept", "magic": "concept", "system": "concept",
        "spell": "spell", "ability": "spell",
        "creature": "creature", "monster": "creature", "beast": "creature",
        "timeline": "timeline", "chronology": "timeline",
    }
    return type_map.get(s, s)  # Return as-is if not in map


def normalize_version(value: Any) -> str:
    """Extract semantic version from raw version string.

    Args:
        value: Raw version (e.g., 'v1.2', '2.0.1', 'rev3')

    Returns:
        Semantic version string (e.g., '1.2', '2.0.1'). Defaults to '1.0'.
    """
    if value is None:
        return "1.0"
    match = re.search(r'v?(\d+(?:\.\d+)*)', str(value))
    return match.group(1) if match else "1.0"


def map_frontmatter(raw_frontmatter: Dict[str, Any]) -> Dict[str, Any]:
    """Map raw frontmatter keys to normalized canonical form.

    Iterates through FIELD_MAP to find matching keys in raw_frontmatter,
    applies appropriate normalization, and returns a clean metadata dict.

    Args:
        raw_frontmatter: Dict with potentially inconsistent keys

    Returns:
        Dict with normalized keys and values. Unmapped keys preserved
        with 'raw_' prefix.
    """
    normalized: Dict[str, Any] = {}
    mapped_raw_keys: set = set()

    for canonical_key, variations in FIELD_MAP.items():
        for variation in variations:
            if variation in raw_frontmatter:
                raw_value = raw_frontmatter[variation]
                mapped_raw_keys.add(variation)

                # Apply appropriate normalization
                if canonical_key == "canon":
                    normalized[canonical_key] = normalize_canon_status(raw_value)
                elif canonical_key == "authority_layer":
                    normalized[canonical_key] = normalize_authority_layer(raw_value)
                elif canonical_key == "type":
                    normalized[canonical_key] = normalize_entity_type(raw_value)
                elif canonical_key == "version":
                    normalized[canonical_key] = normalize_version(raw_value)
                else:
                    normalized[canonical_key] = raw_value

                break  # Use first matching variation

    # Preserve unmapped keys with 'raw_' prefix for diagnostics
    for key, value in raw_frontmatter.items():
        if key not in mapped_raw_keys:
            normalized[f"raw_{key}"] = value

    return normalized


# ============================================================================
# VERIFICATION
# ============================================================================

print("✓ Field Mapping & Normalization Functions Loaded\n")
print(f"  Canonical fields: {len(FIELD_MAP)}")
for key, variations in FIELD_MAP.items():
    print(f"    {key:20s} ← {variations}")

print(f"\n  Normalization functions: canon_status, authority_layer, entity_type, version")
print("✓ Ready to normalize corpus frontmatter.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 06: Corpus Loading

Loads all 25 lore documents from the D-20 test corpus directory.
Parses YAML frontmatter and Markdown body from each file.
Validates loaded documents against ground-truth query expectations.

Identical to D-21 Cell 06 and D-22 Cell 06 for consistency.

Outputs:
  - corpus: List[LoreDocument] (all loaded documents)
  - filename_index: Dict[str, LoreDocument] (fast lookup by filename)
  - Validation report: which ground-truth expected files are present/missing
"""

from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional
from pathlib import Path
from tqdm import tqdm
import yaml # Added missing import
import re # Added missing import for regex in normalization

# Path constant needed for corpus loading
CORPUS_DIR = Path("./corpus")

# ============================================================================
# FIELD MAP: maps canonical key → list of possible raw frontmatter key names
# (Moved from D-23 Cell 05)
# ============================================================================

FIELD_MAP: Dict[str, List[str]] = {
    "type":              ["type", "entity_type", "document_type", "doc_type", "category"],
    "name":              ["name", "entity_name", "title", "subject"],
    "canon":             ["canon", "canon_status", "canonical", "is_canonical"],
    "authority_layer":   ["authority_layer", "authority", "authority_level"],
    "era":               ["era", "age", "time_period", "historical_era", "eras"],
    "domain_layer":      ["domain_layer", "domain", "lore_category"],
    "related_entities":  ["related_entities", "relationships", "relations", "links", "see_also"],
    "version":           ["version", "doc_version", "revision"],
    "tags":              ["tags", "keywords", "labels"],
    "description":       ["description", "summary", "blurb"],
}

def normalize_canon_status(value: Any) -> str:
    """Normalize canon/canonical status to 'true' or 'false'.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "false"
    if isinstance(value, bool):
        return "true" if value else "false"
    s = str(value).lower().strip()
    if s in ("true", "yes", "1", "canonical", "canon"):
        return "true"
    return "false"

def normalize_authority_layer(value: Any) -> str:
    """Normalize authority layer to canonical form.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    if s in ("primary", "official", "canonical", "1"):
        return "primary"
    if s in ("secondary", "semi-official", "semi-canon", "2"):
        return "secondary"
    if s in ("tertiary", "unofficial", "fan", "apocryphal", "3"):
        return "tertiary"
    return "unknown"

def normalize_entity_type(value: Any) -> str:
    """Normalize entity type to canonical form.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "unknown"
    s = str(value).lower().strip()
    type_map = {
        "character": "character", "char": "character", "npc": "character", "person": "character",
        "faction": "faction", "org": "faction", "organization": "faction", "group": "faction",
        "location": "location", "place": "location", "region": "location", "area": "location",
        "event": "event", "battle": "event", "war": "event",
        "item": "item", "artifact": "item", "object": "item", "weapon": "item",
        "concept": "concept", "magic": "concept", "system": "concept",
        "spell": "spell", "ability": "spell",
        "creature": "creature", "monster": "creature", "beast": "creature",
        "timeline": "timeline", "chronology": "timeline",
    }
    return type_map.get(s, s)

def normalize_version(value: Any) -> str:
    """Extract semantic version from raw version string.
    (Moved from D-23 Cell 05)"""
    if value is None:
        return "1.0"
    match = re.search(r'v?(\d+(?:\.\d+)*)', str(value))
    return match.group(1) if match else "1.0"

def map_frontmatter(raw_frontmatter: Dict[str, Any]) -> Dict[str, Any]:
    """Map raw frontmatter keys to normalized canonical form.
    (Moved from D-23 Cell 05)"""
    normalized: Dict[str, Any] = {}
    mapped_raw_keys: set = set()

    for canonical_key, variations in FIELD_MAP.items():
        for variation in variations:
            if variation in raw_frontmatter:
                raw_value = raw_frontmatter[variation]
                mapped_raw_keys.add(variation)

                if canonical_key == "canon":
                    normalized[canonical_key] = normalize_canon_status(raw_value)
                elif canonical_key == "authority_layer":
                    normalized[canonical_key] = normalize_authority_layer(raw_value)
                elif canonical_key == "type":
                    normalized[canonical_key] = normalize_entity_type(raw_value)
                elif canonical_key == "version":
                    normalized[canonical_key] = normalize_version(raw_value)
                else:
                    normalized[canonical_key] = raw_value

                break

    for key, value in raw_frontmatter.items():
        if key not in mapped_raw_keys:
            normalized[f"raw_{key}"] = value

    return normalized

@dataclass
class LoreDocument:
    """A single document from the FractalRecall corpus.

    Attributes:
        filename: Filename (e.g., 'iron_covenant.md')
        frontmatter: Raw YAML frontmatter dict (before normalization)
        body: Markdown body text (after frontmatter)
        metadata: Normalized metadata dict (output of map_frontmatter)
    """
    filename: str
    frontmatter: Dict[str, Any]
    body: str
    metadata: Dict[str, Any]


def parse_lore_file(filepath: Path) -> Tuple[Dict[str, Any], str]:
    """Parse a lore document into frontmatter and body.

    Expects format:
        ---
        key: value
        ---
        # Markdown body

    Args:
        filepath: Path to .md file

    Returns:
        Tuple of (frontmatter_dict, body_str)
    """
    try:
        content = filepath.read_text(encoding="utf-8")
    except Exception as e:
        print(f"  ✗ Error reading {filepath.name}: {e}")
        return {}, ""

    # Split on YAML frontmatter delimiters
    if content.startswith("---"):
        parts = content.split("---", 2)
        if len(parts) >= 3:
            try:
                frontmatter = yaml.safe_load(parts[1]) or {}
            except yaml.YAMLError as e:
                print(f"  ☢ YAML parse error in {filepath.name}: {e}")
                frontmatter = {}
            body = parts[2].strip()
        else:
            frontmatter = {}
            body = content
    else:
        frontmatter = {}
        body = content

    if not isinstance(frontmatter, dict):
        frontmatter = {}

    return frontmatter, body


def estimate_tokens(text: str) -> int:
    """Estimate token count using word-count heuristic.

    Heuristic: tokens ≈ word_count × 1.3
    This is a rough approximation; actual count varies by tokenizer.
    Consistent with D-21/D-22 for fair comparison.

    Args:
        text: Input text string

    Returns:
        Approximate token count (minimum 1)
    """
    if not text or not text.strip():
        return 0
    word_count = len(text.split())
    return max(1, int(word_count * 1.3))

def load_corpus(corpus_dir: Path) -> List[LoreDocument]:
    """Load all .md and .yaml files from corpus directory.

    Args:
        corpus_dir: Path to D-20 test corpus

    Returns:
        List of LoreDocument objects, sorted by filename
    """
    if not corpus_dir.exists():
        print(f"✗ Corpus directory not found: {corpus_dir}")
        print(f"  Ensure D-20 test corpus is available at this path.")
        return []

    # Collect all candidate files
    files = sorted(corpus_dir.glob("*.md"))

    print(f"Loading {len(files)} files from {corpus_dir}...")
    documents = []

    for filepath in tqdm(files, desc="Parsing documents"):
        frontmatter, body = parse_lore_file(filepath)
        metadata = map_frontmatter(frontmatter)

        # Add derived fields
        metadata["filepath"] = str(filepath)
        metadata["filename"] = filepath.name

        doc = LoreDocument(
            filename=filepath.name,
            frontmatter=frontmatter,
            body=body,
            metadata=metadata,
        )
        documents.append(doc)

    return documents


# ============================================================================
# LOAD CORPUS
# ============================================================================

corpus = load_corpus(CORPUS_DIR)

print(f"\n✓ Corpus Loaded: {len(corpus)} documents\n")

# Build filename index for fast lookup
filename_index: Dict[str, LoreDocument] = {doc.filename: doc for doc in corpus}

# ============================================================================
# GROUND-TRUTH VALIDATION
# ============================================================================

# Moved from D-23 Cell 04: Ground-Truth Query Set (Aligned with D-21/D-22)
@dataclass
class GroundTruthQuery:
    """A single ground-truth query with expected relevant documents.

    Attributes:
        query_id: Unique query identifier (e.g., "Q-01")
        query_text: Natural language query
        query_type: One of SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY
        expected_filenames: List of corpus filenames expected to be relevant
        relevance_scores: Dict mapping filename → relevance grade (1-3)
    """
    query_id: str
    query_text: str
    query_type: str
    expected_filenames: List[str]
    relevance_scores: Dict[str, int]

# ============================================================================
# FULL QUERY SET: 36 queries (identical to D-21/D-22)
# ============================================================================

# SINGLE_HOP QUERIES (11 total)
# These test direct lookups of facts, entities, and concepts

QUERIES = [
    {
        "query_id": "Q-01",
        "type": "SINGLE_HOP",
        "text": "What is the Echo-Cant communication system?",
        "relevant_docs": ["000-codex_echo-cant.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-02",
        "type": "SINGLE_HOP",
        "text": "What are the defining characteristics of the Void-Marked?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-03",
        "type": "SINGLE_HOP",
        "text": "Describe the Harrow-Sick condition and its effects.",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_medical-phenomena-entry.md"],
    },
    {
        "query_id": "Q-04",
        "type": "SINGLE_HOP",
        "text": "What is the ODIN Protocol?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-05",
        "type": "SINGLE_HOP",
        "text": "What is the \u00c6ther-Weave operating system?",
        "relevant_docs": ["standalone_aether-weave-os.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-06",
        "type": "SINGLE_HOP",
        "text": "Who are the Scavenger-Barons and what do they do?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-07",
        "type": "SINGLE_HOP",
        "text": "What is the Nine-Tiers architectural framework?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "000-resources_comprehensive-glossary.md"],
    },
    {
        "query_id": "Q-27",
        "type": "SINGLE_HOP",
        "text": "What are the primary functions of the Warden-Host?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-28",
        "type": "SINGLE_HOP",
        "text": "Define the Glitch in the context of Aethelgard's history.",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-30",
        "type": "SINGLE_HOP",
        "text": "What is the Spell-Lock system?",
        "relevant_docs": ["000-codex_spell-lock.md", "standalone_aether-weave-os.md"],
    },
    {
        "query_id": "Q-34",
        "type": "SINGLE_HOP",
        "text": "Describe the Weir-Bone material and its properties.",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "000-resources_comprehensive-glossary.md"],
    },

    # MULTI_HOP QUERIES (8 total)
    # These require traversing relationships between multiple entities/concepts

    {
        "query_id": "Q-08",
        "type": "MULTI_HOP",
        "text": "How do Iron-Bane and God-Sleeper theological positions on Svin-fylking differ?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db02-wb_svin-fylking-assembled-entry.md"],
    },
    {
        "query_id": "Q-09",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Harrow-Sick and the Void-Marked?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-10",
        "type": "MULTI_HOP",
        "text": "How do the Scavenger-Barons use Spell-Lock in their operations?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "000-codex_spell-lock.md", "db03-dc_contract-dispute-case-study.md"],
    },
    {
        "query_id": "Q-11",
        "type": "MULTI_HOP",
        "text": "Explain the conflict between Warden-Host sanctuary protocols and Iron-Bane territorial claims.",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-12",
        "type": "MULTI_HOP",
        "text": "How does the ODIN Protocol interact with the \u00c6ther-Weave OS?",
        "relevant_docs": ["000-codex_odin-protocol.md", "standalone_aether-weave-os.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-29",
        "type": "MULTI_HOP",
        "text": "What role does the Echo-Cant system play in maintaining Warden-Host operations?",
        "relevant_docs": ["000-codex_echo-cant.md", "db02-wb_warden-host-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-31",
        "type": "MULTI_HOP",
        "text": "How do Void-Marked and Weir-Bone materials interact in salvage contexts?",
        "relevant_docs": ["db02-wb_void-marked-assembled-entry.md", "db02-wb_weir-bone-assembled-entry.md", "db03-dc_salvage-operations-manual.md"],
    },
    {
        "query_id": "Q-35",
        "type": "MULTI_HOP",
        "text": "What is the relationship between the Nine-Tiers architecture and Svin-fylking religious doctrine?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },

    # AUTHORITY QUERIES (5 total)
    # These test knowledge of canonical vs. draft vs. superseded information

    {
        "query_id": "Q-13",
        "type": "AUTHORITY",
        "text": "What is the canonical explanation for how the Glitch occurred?",
        "relevant_docs": ["000-resources_comprehensive-glossary.md", "standalone_nine-tiers-architecture.md"],
    },
    {
        "query_id": "Q-14",
        "type": "AUTHORITY",
        "text": "What are the established facts about the God-Sleeper movement origins?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-15",
        "type": "AUTHORITY",
        "text": "Which interpretations of Harrow-Sick etiology are considered canon?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md"],
    },
    {
        "query_id": "Q-16",
        "type": "AUTHORITY",
        "text": "What is the official Warden-Host stance on Void-Marked rights?",
        "relevant_docs": ["db02-wb_warden-host-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-17",
        "type": "AUTHORITY",
        "text": "According to published sources, what materials constitute a valid Spell-Lock?",
        "relevant_docs": ["000-codex_spell-lock.md", "000-resources_comprehensive-glossary.md"],
    },

    # TEMPORAL QUERIES (6 total)
    # These test time-based historical retrieval across the Aethelgard timeline

    {
        "query_id": "Q-18",
        "type": "TEMPORAL",
        "text": "What major events happened in the first century after the Glitch (Year 0-100 PG)?",
        "relevant_docs": ["db03-dc_jotun-reader-chronology.md", "db03-dc_sanctuary-establishment-record.md"],
    },
    {
        "query_id": "Q-19",
        "type": "TEMPORAL",
        "text": "When was the Warden-Host sanctuary established and what precipitated it?",
        "relevant_docs": ["db03-dc_sanctuary-establishment-record.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-20",
        "type": "TEMPORAL",
        "text": "Trace the chronological development of Iron-Bane theological doctrine.",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-21",
        "type": "TEMPORAL",
        "text": "What is the timeline of major salvage discoveries in the Aethelgard region?",
        "relevant_docs": ["db03-dc_salvage-operations-manual.md", "db03-dc_jotun-reader-chronology.md"],
    },
    {
        "query_id": "Q-22",
        "type": "TEMPORAL",
        "text": "When did the God-Sleeper movement gain significant political influence?",
        "relevant_docs": ["db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-23",
        "type": "TEMPORAL",
        "text": "Describe the sequence of events in the Scavenger-Baron contract dispute.",
        "relevant_docs": ["db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },

    # EXPLORATORY QUERIES (6 total)
    # These test open-ended discovery of related concepts and themes

    {
        "query_id": "Q-24",
        "type": "EXPLORATORY",
        "text": "What are the major political tensions in post-Glitch Aethelgard?",
        "relevant_docs": ["db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md", "db03-dc_contract-dispute-case-study.md", "db02-wb_scavenger-barons-assembled-entry.md"],
    },
    {
        "query_id": "Q-25",
        "type": "EXPLORATORY",
        "text": "How do different factions view the technological salvage efforts?",
        "relevant_docs": ["db02-wb_scavenger-barons-assembled-entry.md", "db03-dc_salvage-operations-manual.md", "db03-dc_iron-bane-theological-analysis.md", "db03-dc_god-sleeper-operational-doctrine.md"],
    },
    {
        "query_id": "Q-26",
        "type": "EXPLORATORY",
        "text": "What medical and physiological mysteries remain unsolved in Aethelgard?",
        "relevant_docs": ["000-codex_harrow-sick.md", "db03-dc_medical-causality-research.md", "db02-wb_void-marked-assembled-entry.md"],
    },
    {
        "query_id": "Q-32",
        "type": "EXPLORATORY",
        "text": "What are the intersections between religious doctrine and technological systems in Aethelgard?",
        "relevant_docs": ["standalone_nine-tiers-architecture.md", "db02-wb_svin-fylking-assembled-entry.md", "000-codex_odin-protocol.md", "db03-dc_historical-theological-survey.md"],
    },
    {
        "query_id": "Q-33",
        "type": "EXPLORATORY",
        "text": "How do material properties (Weir-Bone, Void-Marked) influence cultural practices?",
        "relevant_docs": ["db02-wb_weir-bone-assembled-entry.md", "db02-wb_void-marked-assembled-entry.md", "db02-wb_warden-host-assembled-entry.md"],
    },
    {
        "query_id": "Q-36",
        "type": "EXPLORATORY",
        "text": "What gaps exist in the documented understanding of Aethelgard's pre-Glitch history?",
        "relevant_docs": ["db03-dc_historical-theological-survey.md", "db03-dc_jotun-reader-chronology.md", "000-resources_comprehensive-glossary.md"],
    },
]

# ============================================================================
# BUILD GROUND_TRUTH_QUERIES from QUERIES (for compatibility with Cell 14)
# ============================================================================

GROUND_TRUTH_QUERIES: List[GroundTruthQuery] = []
for q in QUERIES:
    # Build uniform relevance scores (all docs score 2 by default)
    relevance = {doc: 2 for doc in q["relevant_docs"]}
    GROUND_TRUTH_QUERIES.append(GroundTruthQuery(
        query_id=q["query_id"],
        query_text=q["text"],
        query_type=q["type"],
        expected_filenames=q["relevant_docs"],
        relevance_scores=relevance,
    ))

# ============================================================================
# VALIDATION
# ============================================================================

query_type_counts = {}
for q in QUERIES:
    query_type_counts[q["type"]] = query_type_counts.get(q["type"], 0) + 1

assert len(QUERIES) == 36, f"Expected 36 queries, got {len(QUERIES)}"
assert query_type_counts.get("SINGLE_HOP", 0) == 11, f"Expected 11 SINGLE_HOP"
assert query_type_counts.get("MULTI_HOP", 0) == 8, f"Expected 8 MULTI_HOP"
assert query_type_counts.get("AUTHORITY", 0) == 5, f"Expected 5 AUTHORITY"
assert query_type_counts.get("TEMPORAL", 0) == 6, f"Expected 6 TEMPORAL"
assert query_type_counts.get("EXPLORATORY", 0) == 6, f"Expected 6 EXPLORATORY"

print(f"✓ All 36 ground-truth queries validated (aligned with D-21/D-22)")
print(f"  Distribution: {query_type_counts}")
print(f"  GROUND_TRUTH_QUERIES: {len(GROUND_TRUTH_QUERIES)} GroundTruthQuery objects")

# ============================================================================
# GROUND-TRUTH VALIDATION
# ============================================================================

print("Ground-Truth Validation:")
all_expected = set()
for q in GROUND_TRUTH_QUERIES:
    all_expected.update(q.expected_filenames)

missing = all_expected - set(filename_index.keys())
found = all_expected & set(filename_index.keys())

if missing:
    print(f"  ☢ {len(missing)} expected files NOT in corpus:")
    for fn in sorted(missing):
        print(f"      - {fn}")
else:
    print(f"  ✓ All {len(found)} expected files found in corpus")

# ============================================================================
# CORPUS SUMMARY
# ============================================================================

print(f"\nCorpus Summary:")
total_body_tokens = sum(estimate_tokens(doc.body) for doc in corpus)
print(f"  Documents:            {len(corpus)}")
print(f"  Total body tokens:    {total_body_tokens:,}")
print(f"  Avg tokens/doc:       {total_body_tokens // max(len(corpus), 1):,}")

# Metadata coverage
meta_keys = set()
for doc in corpus:
    meta_keys.update(doc.metadata.keys())
print(f"  Unique metadata keys: {len(meta_keys)}")

# Key field coverage
for field in ["type", "name", "canon", "era", "related_entities"]:
    present = sum(1 for doc in corpus if doc.metadata.get(field))
    pct = 100 * present / max(len(corpus), 1)
    print(f"  {field:20s}: {present}/{len(corpus)} ({pct:.0f}%)")

print("\n✓ Corpus ready for chunking.")


## Methodology: Multi-Layer Enrichment & GO/NO-GO Decision

### Research Hypothesis

**H1 (Primary)**: Multi-layer enrichment (D-23) significantly outperforms single-layer enrichment (D-22) in at least 2 of 4 retrieval metrics.

**H2 (Secondary)**: Entity and Relational layers provide the largest marginal improvement beyond the single-layer prefix.

**H3 (Tertiary)**: Authority-sensitive queries (Q-01 to Q-12) and temporal queries (Q-13 to Q-24) benefit more from multi-layer enrichment than factual queries (Q-25 to Q-36).

### The 8 Context Layers

Each chunk receives a multi-layer prefix constructed from these layers in order:

| # | Layer | Varies By | Token Budget | Source Field |
|---|-------|-----------|-------------|--------------|
| 1 | Corpus | Constant | ~5 | Hardcoded: CORPUS_LABEL |
| 2 | Domain | Document | ~8 | metadata.type → DOMAIN_CATEGORY_MAP |
| 3 | Entity | Document | ~8 | metadata.name |
| 4 | Authority | Document | ~8 | metadata.canon → authority mapping |
| 5 | Temporal | Document | ~12 | metadata.era (list → "X and Y") |
| 6 | Relational | Document | ~15-30 | metadata.related_entities (parsed) |
| 7 | Section | **Chunk** | ~10 | chunk.section_heading |
| 8 | Content | **Chunk** | variable | chunk.text (raw content) |

**Key difference from D-22**: The Section layer varies per chunk (not per document), so every chunk gets a unique prefix. This is more granular than D-22's static document-level prefix.

### Token Budget Impact

| Model | Max Tokens | Prefix Reserve | Content Budget | Prefix % |
|-------|-----------|---------------|---------------|----------|
| v2-moe | 512 | 100 | 250 | ~29% |
| **v1.5** | 8192 | 150 | 450 | ~25% |
| bge-m3 | 8192 | 150 | 874 | ~15% |

D-23 chunks are **shorter** than D-21/D-22 to accommodate the multi-layer prefix within the model's context window.

### 3-Way Comparison Design

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 08: Hybrid Chunking Engine (Round 4 — D-21 Algorithm Port)

CRITICAL FIX (Round 4):
  Rounds 1-3 used a reimplemented chunker that omitted:
    1. min_chunk_tokens=128 merge logic → produced 1,233 tiny chunks
    2. #{1,6} heading regex (used #{2,4} instead)
  
  This cell now uses D-21 Cell 8's EXACT algorithm.
  The ONLY modification: prefix_reserve subtracted from token budget.

Ported from D-21 Cell 8:
  - split_into_sections(): #{1,6} heading regex
  - sliding_window_split(): overlapping windows for oversized sections
  - chunk_document(): min_chunk_tokens merge logic preserved
"""

import re
from typing import List, Tuple, Optional, Dict, Any

# ============================================================================
# CONSTANTS — Match D-21 exactly
# ============================================================================

MIN_CHUNK_TOKENS = 128  # D-21: config.min_chunk_tokens = 128
OVERLAP_TOKENS = 150    # D-21: config.overlap_tokens = 150

# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class Chunk:
    """A single text chunk with metadata."""
    chunk_id: str
    doc_filename: str
    section_heading: Optional[str]
    text: str
    token_count_approx: int
    metadata: Dict[str, Any]


# ============================================================================
# TOKEN ESTIMATION — Identical to D-21
# ============================================================================

def estimate_tokens(text: str) -> int:
    """Approximate token count using word_count * 1.3 heuristic.
    Identical to D-21 Cell 8."""
    if not text or not text.strip():
        return 0
    return max(1, int(len(text.split()) * 1.3))


# ============================================================================
# HEADING EXTRACTION — D-21's exact regex and logic
# ============================================================================

def split_into_sections(body: str) -> List[Tuple[str, str]]:
    """Split a markdown body into (heading, content) pairs.

    PORTED FROM D-21 CELL 8 — EXACT COPY.
    Splits on lines starting with # through ###### (levels 1-6).
    The first section may have heading="" if text precedes the first heading.
    """
    # D-21's exact regex: matches ALL heading levels 1-6
    pattern = r'^(#{1,6}\s+.+)$'
    parts = re.split(pattern, body, flags=re.MULTILINE)

    sections = []
    current_heading = ""
    current_content = ""

    for part in parts:
        part_stripped = part.strip()
        if re.match(r'^#{1,6}\s+', part_stripped):
            # Save previous section if it has content
            if current_content.strip():
                sections.append((current_heading, current_content.strip()))
            current_heading = part_stripped
            current_content = ""
        else:
            current_content += part

    # Don't forget the last section
    if current_content.strip():
        sections.append((current_heading, current_content.strip()))

    # If no sections found, treat entire body as one section
    if not sections and body.strip():
        sections = [("", body.strip())]

    return sections


# ============================================================================
# SLIDING WINDOW — D-21's exact logic
# ============================================================================

def sliding_window_split(text: str, max_tokens: int, overlap_tokens: int) -> List[str]:
    """Split text into overlapping windows when it exceeds max_tokens.

    PORTED FROM D-21 CELL 8 — EXACT COPY.
    Uses word-level splitting with token estimation.
    """
    words = text.split()
    # Convert token limits to approximate word counts
    max_words = int(max_tokens / 1.3)
    overlap_words = int(overlap_tokens / 1.3)
    step = max(1, max_words - overlap_words)

    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + max_words]
        chunks.append(" ".join(chunk_words))
        i += step
        if i + max_words >= len(words) and i < len(words):
            # Last chunk: take remaining words
            chunks.append(" ".join(words[i:]))
            break

    return chunks


# ============================================================================
# DOCUMENT CHUNKING — D-21's exact algorithm + prefix_reserve
# ============================================================================

def chunk_document(
    doc: LoreDocument,
    max_chunk_tokens: int,
    prefix_reserve: Optional[int] = None,
) -> List[Chunk]:
    """Chunk a document using D-21's hybrid heading + sliding-window approach.

    PORTED FROM D-21 CELL 8 with ONE modification:
      prefix_reserve is subtracted from the token budget so the
      enrichment prefix fits within max_chunk_tokens.

    D-21 behavior preserved:
      - split_into_sections() with #{1,6} heading regex
      - MIN_CHUNK_TOKENS=128 merge logic for short sections
      - sliding_window_split() for oversized sections
      - OVERLAP_TOKENS=150
    """
    if prefix_reserve is None:
        prefix_reserve = MODEL_CONFIG.prefix_reserve_tokens

    # The ONLY difference from D-21: subtract prefix_reserve from budget
    # D-21 effective_max = config.max_chunk_tokens (1024, full budget)
    # D-23 effective_max = max_chunk_tokens - prefix_reserve (1024 - 100 = 924)
    effective_max = max_chunk_tokens - prefix_reserve

    chunks = []
    chunk_counter = 0

    # Step 1: Split body into sections by markdown headings
    # D-21's split_into_sections: #{1,6} regex
    sections = split_into_sections(doc.body)

    for heading, content in sections:
        token_est = estimate_tokens(content)

        if token_est <= effective_max:
            # Section fits within limit — check if it meets minimum
            if token_est >= MIN_CHUNK_TOKENS:
                # Normal-sized section — create single chunk
                chunk_counter += 1
                chunks.append(Chunk(
                    chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                    doc_filename=doc.filename,
                    section_heading=heading if heading else None,
                    text=content,
                    token_count_approx=token_est,
                    metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                ))
            else:
                # *** D-21's MERGE LOGIC — was missing in D-23 Rounds 1-3 ***
                # Section below MIN_CHUNK_TOKENS — merge with previous chunk
                if chunks:
                    prev = chunks[-1]
                    merged_text = prev.text + "\n\n" + content
                    merged_tokens = estimate_tokens(merged_text)
                    if merged_tokens <= effective_max:
                        # Merge into previous chunk
                        chunks[-1] = Chunk(
                            chunk_id=prev.chunk_id,
                            doc_filename=prev.doc_filename,
                            section_heading=prev.section_heading,
                            text=merged_text,
                            token_count_approx=merged_tokens,
                            metadata=prev.metadata.copy() if hasattr(prev.metadata, 'copy') else dict(prev.metadata),
                        )
                    else:
                        # Can't merge (would exceed budget) — keep as small chunk
                        chunk_counter += 1
                        chunks.append(Chunk(
                            chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                            doc_filename=doc.filename,
                            section_heading=heading if heading else None,
                            text=content,
                            token_count_approx=token_est,
                            metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                        ))
                else:
                    # First chunk and it's small — keep it anyway
                    chunk_counter += 1
                    chunks.append(Chunk(
                        chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                        doc_filename=doc.filename,
                        section_heading=heading if heading else None,
                        text=content,
                        token_count_approx=token_est,
                        metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                    ))
        else:
            # Section too large — apply sliding window (D-21's logic)
            sub_texts = sliding_window_split(
                content,
                effective_max,
                OVERLAP_TOKENS,
            )
            for sub_text in sub_texts:
                chunk_counter += 1
                chunks.append(Chunk(
                    chunk_id=f"{doc.filename}#chunk_{chunk_counter:03d}",
                    doc_filename=doc.filename,
                    section_heading=heading if heading else None,
                    text=sub_text,
                    token_count_approx=estimate_tokens(sub_text),
                    metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
                ))

    # Edge case: document with no body at all (from D-21)
    if not chunks and doc.body.strip():
        chunks.append(Chunk(
            chunk_id=f"{doc.filename}#chunk_001",
            doc_filename=doc.filename,
            section_heading=None,
            text=doc.body.strip(),
            token_count_approx=estimate_tokens(doc.body),
            metadata=doc.metadata.copy() if hasattr(doc.metadata, 'copy') else dict(doc.metadata),
        ))

    return chunks


# ============================================================================
# CONFIGURATION SUMMARY
# ============================================================================

effective = MODEL_CONFIG.max_chunk_tokens - MODEL_CONFIG.prefix_reserve_tokens
print("=" * 70)
print("CHUNKING ENGINE — D-21 Algorithm Port (Round 4 Fix)")
print("=" * 70)
print(f"\n  Model:                {MODEL_CONFIG.name}")
print(f"  Max chunk tokens:     {MODEL_CONFIG.max_chunk_tokens}")
print(f"  Prefix reserve:       {MODEL_CONFIG.prefix_reserve_tokens} tokens")
print(f"  Effective for content:{effective} tokens")
print(f"  Min chunk tokens:     {MIN_CHUNK_TOKENS}  [D-21 match]")
print(f"  Overlap:              {OVERLAP_TOKENS} tokens  [D-21 match]")
print(f"  Heading regex:        #{{1,6}}  [D-21 match]")
print(f"  Token estimator:      word_count x 1.3  [D-21 match]")
print(f"  Merge logic:          ENABLED  [D-21 match — was MISSING in R1-R3]")
print(f"\n  [Round 4] D-21's exact chunking algorithm ported.")
print(f"  Only change: prefix_reserve ({MODEL_CONFIG.prefix_reserve_tokens} tokens) subtracted from budget.")
print(f"  Expected chunk count: ~200-250 (was 1,233 without merge logic)")
print("=" * 70)

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 09: Multi-Layer Enrichment Builder — THE CORE NEW CODE

Implements the full 8-layer context enrichment for FractalRecall.
This is the key difference from D-21 (no enrichment) and D-22 (single-layer).

Architecture:
  - 7 individual layer builder functions (Corpus through Section)
  - Each returns Optional[str] — None means the layer is omitted
  - build_multi_layer_prefix() assembles all layers, filters None, joins with "\\n\\n"
  - build_enriched_chunk() prepends the prefix to a chunk's text

Layer rendering format (from COLAB-SESSION-CONTEXT.md / D-32):
  Corpus: Aethelgard Worldbuilding Corpus v5.0

  Domain: This content is from a faction document in the organizations category.

  Entity: This content describes The Iron Covenant.

  Authority: This content is canonical and authoritative.

  Temporal: The events described span the Third Age and Fourth Age.

  Relationships: founded by Elena Voss; rival of Silver Hand; located in Ashenmoor.

  Section: This content is from the Origins section.

  [chunk text here]

Token budget:
  - Best case (sparse metadata):  ~50 tokens
  - Typical case:                 ~80-100 tokens
  - Worst case (rich relational): ~150 tokens
  - Content budget (v1.5):        ~450 tokens (600 - 150 reserve)

Reference: COLAB-SESSION-CONTEXT.md §"The 8 Context Layers"; D-22 Cell 09 (comparison)
"""

# ============================================================================
# DOMAIN CATEGORY MAPPING
# ============================================================================
# Maps document entity types to broader domain categories for the Domain layer.
# Used by build_domain_layer() to produce richer context than just the raw type.

DOMAIN_CATEGORY_MAP: Dict[str, str] = {
    "faction":    "organizations",
    "character":  "individuals",
    "location":   "geography",
    "event":      "history",
    "item":       "artifacts",
    "concept":    "metaphysics",
    "spell":      "magic system",
    "creature":   "bestiary",
    "region":     "geography",
    "timeline":   "chronology",
}


# ============================================================================
# LAYER BUILDER FUNCTIONS
# ============================================================================
# Each function builds one context layer.
# Returns Optional[str]: a rendered layer string, or None to omit the layer.
# None layers are excluded from the final prefix (not rendered as empty lines).

def build_corpus_layer() -> str:
    """Build the Corpus layer (Layer 1).

    Always returns a value — the Corpus layer is never omitted.
    Uses the CORPUS_LABEL constant defined in Cell 03.

    Returns:
        str: "Corpus: {CORPUS_LABEL}"

    Example:
        >>> build_corpus_layer()
        'Corpus: Aethelgard Worldbuilding Corpus v5.0'
    """
    return f"Corpus: {CORPUS_LABEL}"


def build_domain_layer(metadata: Dict[str, Any]) -> str:
    """Build the Domain layer (Layer 2).

    Maps the document type to a broader domain category via DOMAIN_CATEGORY_MAP.
    Always returns a value (defaults to 'general' for unknown types).

    Args:
        metadata: Normalized document metadata with 'type' field.

    Returns:
        str: "Domain: This content is from a {type} document in the {category} category."

    Example:
        >>> build_domain_layer({"type": "faction"})
        'Domain: This content is from a faction document in the organizations category.'
    """
    doc_type = str(metadata.get("type", "unknown")).lower().strip()
    category = DOMAIN_CATEGORY_MAP.get(doc_type, "general")
    return f"Domain: This content is from a {doc_type} document in the {category} category."


def build_entity_layer(metadata: Dict[str, Any]) -> Optional[str]:
    """Build the Entity layer (Layer 3).

    Returns None if the entity name is missing or "Unknown", causing this layer
    to be omitted from the prefix entirely.

    Args:
        metadata: Normalized metadata with 'name' field.

    Returns:
        Optional[str]: "Entity: This content describes {name}." or None.

    Examples:
        >>> build_entity_layer({"name": "The Iron Covenant"})
        'Entity: This content describes The Iron Covenant.'

        >>> build_entity_layer({"name": "Unknown"})
        None
    """
    name = str(metadata.get("name", "")).strip()
    if not name or name.lower() == "unknown":
        return None
    return f"Entity: This content describes {name}."


def build_authority_layer(metadata: Dict[str, Any]) -> str:
    """Build the Authority layer (Layer 4).

    Maps the 'canon' field to a human-readable authority classification.
    Always returns a value (defaults to 'draft' for unrecognized values).

    Mapping:
      - canon=True / "true" / "yes" / "canonical"  → "canonical and authoritative"
      - canon="apocryphal"                          → "apocryphal (non-canonical, speculative)"
      - canon="deprecated"                          → "deprecated and superseded"
      - anything else                               → "draft (not yet canonical)"

    Args:
        metadata: Normalized metadata with 'canon' field.

    Returns:
        str: "Authority: This content is {authority_text}."

    Examples:
        >>> build_authority_layer({"canon": "true"})
        'Authority: This content is canonical and authoritative.'

        >>> build_authority_layer({"canon": "apocryphal"})
        'Authority: This content is apocryphal (non-canonical, speculative).'
    """
    canon = metadata.get("canon", "")

    # Normalize to string
    if isinstance(canon, bool):
        canon_str = "true" if canon else "false"
    else:
        canon_str = str(canon).lower().strip()

    # Map to authority text
    if canon_str in ("true", "yes", "canonical"):
        authority_text = "canonical and authoritative"
    elif canon_str == "apocryphal":
        authority_text = "apocryphal (non-canonical, speculative)"
    elif canon_str == "deprecated":
        authority_text = "deprecated and superseded"
    else:
        authority_text = "draft (not yet canonical)"

    return f"Authority: This content is {authority_text}."


def build_temporal_layer(metadata: Dict[str, Any]) -> Optional[str]:
    """Build the Temporal layer (Layer 5).

    Returns None if no temporal data is available, causing this layer to be omitted.
    Handles both 'era' format (e.g., ["Third Age"]) and 'temporal_markers' format
    (e.g., ["Year 783 PG", "Years 0-122 PG"]) from the Aethelgard corpus.
    [FIX: Round 3 — handle temporal_markers from corpus YAML frontmatter]

    Args:
        metadata: Normalized metadata with 'era' field (list of strings or single string).
            The 'era' key may contain temporal_markers values after FIELD_MAP normalization.

    Returns:
        Optional[str]: Temporal context sentence or None.

    Examples:
        >>> build_temporal_layer({"era": ["Third Age", "Fourth Age"]})
        'Temporal: The events described span the Third Age and Fourth Age.'

        >>> build_temporal_layer({"era": ["Year 783 PG", "Years 0-122 PG"]})
        'Temporal: Events reference Year 783 PG and Years 0-122 PG.'

        >>> build_temporal_layer({})
        None
    """
    eras = metadata.get("era", [])

    # Handle single string
    if isinstance(eras, str):
        eras = [eras] if eras.strip() else []

    # Filter empty values
    eras_clean = [str(e).strip() for e in eras if e and str(e).strip()]

    if not eras_clean:
        return None

    era_text = " and ".join(eras_clean)

    # Detect temporal_markers format (contains "Year" or "PG")
    is_marker_format = any("Year" in e or "PG" in e for e in eras_clean)
    if is_marker_format:
        return f"Temporal: Events reference {era_text}."
    else:
        return f"Temporal: The events described span the {era_text}."


def build_relational_layer(metadata: Dict[str, Any]) -> Optional[str]:
    """Build the Relational layer (Layer 6).

    Parses the relationships from metadata. Handles three formats:
      1. Dict format: [{"target": "...", "type": "..."}] (structured)
      2. Tuple/list format: [("type", "target")] (structured)
      3. String list format: ["Silent Folk", "Echo-Mothers"] (cross_references)
    [FIX: Round 3 — handle string-list cross_references from corpus YAML]

    Also incorporates factions_mentioned and locations_mentioned from raw_
    prefixed keys (preserved by map_frontmatter for unmapped keys).

    Returns None if no relationships are present.

    Args:
        metadata: Normalized metadata with 'related_entities' field and
            optionally 'raw_factions_mentioned' and 'raw_locations_mentioned'.

    Returns:
        Optional[str]: "Related to: {entities}." or None.

    Examples:
        >>> build_relational_layer({"related_entities": ["Silent Folk", "Echo-Mothers"]})
        'Related to: Silent Folk, Echo-Mothers.'

        >>> build_relational_layer({"related_entities": [
        ...     {"target": "characters/elena-voss.md", "type": "founded_by"}]})
        'Related to: Elena Voss (founded by).'
    """
    relationships = metadata.get("related_entities", [])
    factions = metadata.get("raw_factions_mentioned", [])
    locations = metadata.get("raw_locations_mentioned", [])

    rel_parts: List[str] = []

    for rel in relationships:
        # Handle dict format: {"target": "...", "type": "..."}
        if isinstance(rel, dict):
            rel_type = str(rel.get("type", "related to")).replace("_", " ")
            target = str(rel.get("target", ""))
            # Extract human-readable name from file path
            if "/" in target:
                target = target.split("/")[-1]
            if target.endswith(".md"):
                target = target[:-3]
            target_name = " ".join(
                word.capitalize() for word in target.replace("-", " ").replace("_", " ").split()
            )
            if rel_type and target_name:
                rel_parts.append(f"{target_name} ({rel_type})")
        # Handle tuple/list format: (type, target) or [type, target]
        elif isinstance(rel, (tuple, list)) and len(rel) >= 2:
            rel_type = str(rel[0]).replace("_", " ")
            target_name = str(rel[1])
            if rel_type and target_name:
                rel_parts.append(f"{target_name} ({rel_type})")
        # Handle string format: "Silent Folk" (from cross_references)
        elif isinstance(rel, str) and rel.strip():
            rel_parts.append(rel.strip())

    # Add factions and locations as additional relational context
    for faction in (factions if isinstance(factions, list) else []):
        if isinstance(faction, str) and faction.strip() and faction.strip() not in rel_parts:
            rel_parts.append(faction.strip())

    for location in (locations if isinstance(locations, list) else []):
        if isinstance(location, str) and location.strip() and location.strip() not in rel_parts:
            rel_parts.append(location.strip())

    if not rel_parts:
        return None

    return f"Related to: {', '.join(rel_parts)}."


def build_section_layer(section_heading: Optional[str]) -> Optional[str]:
    """Build the Section layer (Layer 7).

    Returns None if no section heading is available. This layer varies per CHUNK
    (not per document), making it the key differentiator from D-22's approach.

    Args:
        section_heading: The Markdown heading of the section this chunk belongs to.

    Returns:
        Optional[str]: "Section: This content is from the {heading} section." or None.

    Examples:
        >>> build_section_layer("Origins")
        'Section: This content is from the Origins section.'

        >>> build_section_layer(None)
        None
    """
    if not section_heading or not str(section_heading).strip():
        return None
    return f"Section: This content is from the {section_heading} section."


# ============================================================================
# MULTI-LAYER PREFIX BUILDER
# ============================================================================

def build_multi_layer_prefix(
    metadata: Dict[str, Any],
    section_heading: Optional[str] = None,
) -> Tuple[str, Dict[str, int]]:
    """Assemble the complete multi-layer enrichment prefix.

    Calls each layer builder in order (Corpus → Domain → Entity → Authority →
    Temporal → Relational → Section). Filters out None results. Joins remaining
    layers with double newlines ("\\n\\n").

    Also produces a token audit dict mapping each present layer to its
    approximate token count, used for token budget analysis in Cell 10.

    Args:
        metadata: Normalized document metadata dict.
        section_heading: Section heading for the Section layer (varies per chunk).

    Returns:
        Tuple of:
          - prefix_text (str): Complete multi-layer prefix joined by "\\n\\n"
          - layer_token_audit (Dict[str, int]): Maps layer_name → token_count

    Example:
        >>> meta = {"type": "faction", "name": "Iron Covenant", "canon": "true"}
        >>> prefix, audit = build_multi_layer_prefix(meta, "Origins")
        >>> print(audit)
        {'Corpus': 6, 'Domain': 12, 'Entity': 8, 'Authority': 7, 'Section': 10}
    """
    # Define builders in layer order
    layer_builders: List[Tuple[str, Any]] = [
        ("Corpus",        lambda: build_corpus_layer()),
        ("Domain",        lambda: build_domain_layer(metadata)),
        ("Entity",        lambda: build_entity_layer(metadata)),
        ("Authority",     lambda: build_authority_layer(metadata)),
        ("Temporal",      lambda: build_temporal_layer(metadata)),
        ("Relationships", lambda: build_relational_layer(metadata)),
        ("Section",       lambda: build_section_layer(section_heading)),
    ]

    layer_token_audit: Dict[str, int] = {}
    prefix_parts: List[str] = []

    for layer_name, builder_fn in layer_builders:
        layer_text = builder_fn()
        if layer_text is not None:
            prefix_parts.append(layer_text)
            layer_token_audit[layer_name] = estimate_tokens(layer_text)

    # Join with double newlines (clear semantic boundary between layers)
    prefix_text = "\n\n".join(prefix_parts)

    return prefix_text, layer_token_audit


# ============================================================================
# ENRICHED CHUNK BUILDER
# ============================================================================

def build_enriched_chunk(
    chunk: Chunk,
    enrichment_type: str = "multi_layer",
) -> Tuple[Chunk, Dict[str, int]]:
    """Build an enriched chunk by prepending the multi-layer prefix.

    Constructs the 8-layer prefix from the chunk's metadata and section heading,
    then prepends it to the chunk text with a "\\n\\n" separator.

    Token overflow is LOGGED as a warning but does NOT raise an exception.
    This allows us to track overflow frequency across the corpus (Cell 10).

    Args:
        chunk: Original Chunk object (raw text, no enrichment)
        enrichment_type: Label for enrichment method (default "multi_layer")

    Returns:
        Tuple of:
          - enriched_chunk (Chunk): New Chunk with enriched text and updated token count
          - layer_token_audit (Dict[str, int]): Token counts per layer

    Example:
        >>> enriched, audit = build_enriched_chunk(some_chunk)
        >>> enriched.token_count_approx  # includes prefix + content
        185
        >>> audit
        {'Corpus': 6, 'Domain': 12, 'Entity': 8, ...}
    """
    # Build the multi-layer prefix
    prefix_text, layer_token_audit = build_multi_layer_prefix(
        chunk.metadata,
        chunk.section_heading,
    )

    # Combine prefix + chunk text with clear separator
    enriched_text = f"{prefix_text}\n\n{chunk.text}"
    enriched_tokens = estimate_tokens(enriched_text)

    # Token overflow safety check (WARNING only — does not raise)
    if enriched_tokens > MODEL_CONFIG.max_chunk_tokens:
        print(
            f"  ⚠ Overflow: {chunk.chunk_id} — "
            f"{enriched_tokens} tokens > {MODEL_CONFIG.max_chunk_tokens} limit "
            f"(prefix={sum(layer_token_audit.values())}, content={chunk.token_count_approx})"
        )

    # Create enriched chunk (same metadata, updated text and token count)
    enriched_chunk = Chunk(
        chunk_id=chunk.chunk_id,
        doc_filename=chunk.doc_filename,
        section_heading=chunk.section_heading,
        text=enriched_text,
        token_count_approx=enriched_tokens,
        metadata=chunk.metadata,
    )

    return enriched_chunk, layer_token_audit


# ============================================================================
# TEST SECTION
# ============================================================================

print("=" * 80)
print("MULTI-LAYER ENRICHMENT BUILDER — TEST")
print("=" * 80)

# Create a sample chunk with rich metadata
sample_metadata = {
    "type": "faction",
    "name": "The Iron Covenant",
    "canon": "true",
    "era": ["Third Age", "Fourth Age"],
    "related_entities": [
        {"target": "characters/elena-voss.md", "type": "founded_by"},
        {"target": "factions/silver-hand.md", "type": "rivalry"},
        {"target": "locations/ashenmoor.md", "type": "located_in"},
    ],
}

sample_chunk = Chunk(
    chunk_id="iron_covenant.md#chunk_001",
    doc_filename="iron_covenant.md",
    section_heading="Origins",
    text=(
        "The Iron Covenant was founded in Year 412 of the Third Age by "
        "Commander Elena Voss. It emerged as a powerful military organization "
        "devoted to maintaining order in the Ashenmoor region."
    ),
    token_count_approx=estimate_tokens(
        "The Iron Covenant was founded in Year 412 of the Third Age by "
        "Commander Elena Voss. It emerged as a powerful military organization "
        "devoted to maintaining order in the Ashenmoor region."
    ),
    metadata=sample_metadata,
)

# Test prefix builder
print("\n[1] Layer-by-layer breakdown:")
prefix_text, layer_audit = build_multi_layer_prefix(sample_metadata, "Origins")
total_prefix_tokens = 0
for layer_name, token_count in layer_audit.items():
    print(f"  {layer_name:15s} | {token_count:3d} tokens")
    total_prefix_tokens += token_count
print(f"  {'─' * 35}")
print(f"  {'TOTAL PREFIX':15s} | {total_prefix_tokens:3d} tokens")

# Test rendered prefix
print(f"\n[2] Rendered prefix ({total_prefix_tokens} tokens):")
print("-" * 80)
print(prefix_text)
print("-" * 80)

# Test enriched chunk
enriched_chunk, _ = build_enriched_chunk(sample_chunk)
print(f"\n[3] Enrichment summary:")
print(f"  Original chunk:  {sample_chunk.token_count_approx} tokens")
print(f"  Prefix overhead: {total_prefix_tokens} tokens")
print(f"  Enriched total:  {enriched_chunk.token_count_approx} tokens")
print(f"  Budget used:     {enriched_chunk.token_count_approx}/{MODEL_CONFIG.max_chunk_tokens} "
      f"({100*enriched_chunk.token_count_approx/MODEL_CONFIG.max_chunk_tokens:.1f}%)")

print("\n✓ Multi-layer enrichment builder initialized and tested.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 10: Chunk, Enrich & Audit Corpus

Applies chunking to all documents, then applies multi-layer enrichment
to each chunk. Tracks per-layer token consumption for budget analysis.

Key differences from D-22 Cell 10:
  1. Uses build_enriched_chunk() with 8-layer prefix (not single-layer)
  2. Tracks per-layer token consumption (layer_token_audits list)
  3. Reports metadata completeness (which optional layers are present)
  4. Warns on overflow but continues (does NOT raise)

Outputs:
  - enriched_chunks: List[Chunk] (enriched with multi-layer prefix)
  - layer_token_audits: List[Dict[str, int]] (per-chunk layer token breakdown)
  - Summary statistics: token overhead, overflow rate, layer presence rates
"""

enriched_chunks: List[Chunk] = []
layer_token_audits: List[Dict[str, Any]] = []

# Track statistics
tokens_before: List[int] = []     # Raw chunk tokens (pre-enrichment)
tokens_after: List[int] = []      # Enriched chunk tokens (post-enrichment)
overflow_count = 0

print("=" * 80)
print("CHUNK, ENRICH & AUDIT CORPUS")
print("=" * 80)
print(f"\n  Model: {MODEL_CONFIG.name}")
print(f"  Max chunk tokens: {MODEL_CONFIG.max_chunk_tokens}")
print(f"  Prefix reserve: {MODEL_CONFIG.prefix_reserve_tokens}")
print(f"  Available for content: {MODEL_CONFIG.max_chunk_tokens - MODEL_CONFIG.prefix_reserve_tokens}")
print()

for doc in tqdm(corpus, desc="Chunk & enrich"):
    # Step 1: Chunk the document (reserves prefix_reserve_tokens for enrichment)
    doc_chunks = chunk_document(doc, MODEL_CONFIG.max_chunk_tokens)

    # Step 2: Enrich each chunk with multi-layer prefix
    for chunk in doc_chunks:
        tokens_before.append(chunk.token_count_approx)

        enriched_chunk, layer_audit = build_enriched_chunk(chunk)

        tokens_after.append(enriched_chunk.token_count_approx)

        # Track overflow
        if enriched_chunk.token_count_approx > MODEL_CONFIG.max_chunk_tokens:
            overflow_count += 1

        enriched_chunks.append(enriched_chunk)

        # Record audit with chunk identification
        audit_record = {
            "chunk_id": chunk.chunk_id,
            "doc_filename": chunk.doc_filename,
            "section_heading": chunk.section_heading or "(none)",
            "tokens_raw": chunk.token_count_approx,
            "tokens_enriched": enriched_chunk.token_count_approx,
            **layer_audit,
        }
        layer_token_audits.append(audit_record)

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print(f"\n✓ Enrichment Complete: {len(enriched_chunks)} chunks\n")

print("TOKEN STATISTICS:")
print("-" * 60)

arr_before = np.array(tokens_before)
arr_after = np.array(tokens_after)
arr_overhead = arr_after - arr_before

print(f"  {'':20s} {'Before':>10s} {'After':>10s} {'Overhead':>10s}")
print(f"  {'Mean':20s} {arr_before.mean():>10.1f} {arr_after.mean():>10.1f} {arr_overhead.mean():>10.1f}")
print(f"  {'Median':20s} {np.median(arr_before):>10.1f} {np.median(arr_after):>10.1f} {np.median(arr_overhead):>10.1f}")
print(f"  {'Max':20s} {arr_before.max():>10d} {arr_after.max():>10d} {arr_overhead.max():>10d}")
print(f"  {'Min':20s} {arr_before.min():>10d} {arr_after.min():>10d} {arr_overhead.min():>10d}")

print(f"\n  Overflow chunks: {overflow_count}/{len(enriched_chunks)} "
      f"({100*overflow_count/max(len(enriched_chunks),1):.1f}%)")

# Overhead distribution
pcts = np.percentile(arr_overhead, [0, 25, 50, 75, 100])
print(f"\n  Prefix overhead distribution:")
print(f"    Min:    {pcts[0]:.0f} tokens")
print(f"    P25:    {pcts[1]:.0f} tokens")
print(f"    Median: {pcts[2]:.0f} tokens")
print(f"    P75:    {pcts[3]:.0f} tokens")
print(f"    Max:    {pcts[4]:.0f} tokens")

# ============================================================================
# LAYER PRESENCE RATES
# ============================================================================

print(f"\nLAYER PRESENCE RATES:")
print("-" * 60)

layer_names = ["Corpus", "Domain", "Entity", "Authority", "Temporal", "Relationships", "Section"]
total = len(layer_token_audits)

for layer in layer_names:
    present = sum(1 for audit in layer_token_audits if layer in audit)
    rate = 100 * present / max(total, 1)
    avg_tokens = np.mean([audit[layer] for audit in layer_token_audits if layer in audit]) if present else 0
    print(f"  {layer:15s}: {present:4d}/{total} ({rate:5.1f}%) — avg {avg_tokens:.1f} tokens")

# ============================================================================
# EXPORT TOKEN AUDIT
# ============================================================================

audit_df = pd.DataFrame(layer_token_audits)
audit_csv_path = OUTPUT_DIR / "d23_layer_token_audits.csv"
audit_df.to_csv(audit_csv_path, index=False)
print(f"\n✓ Token audit exported to {audit_csv_path}")
print(f"  Rows: {len(audit_df)}, Columns: {len(audit_df.columns)}")

print("\n✓ Corpus chunked, enriched, and audited. Ready for embedding.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 11: Embedding & ChromaDB Indexing

Embeds all enriched chunks and indexes them in a ChromaDB collection.
Creates collection d23_multi_layer_{model}.

Same embedding pipeline as D-22 Cell 11, but operating on multi-layer
enriched chunks instead of single-layer enriched chunks.

Supports:
  - sentence-transformers (v2-moe, v1.5): encode with prompt_name="document"
  - FlagEmbedding (bge-m3): BGEM3FlagModel.encode with dense vectors
"""

print("=" * 80)
print("EMBEDDING & CHROMADB INDEXING")
print("=" * 80)

# ============================================================================
# LOAD EMBEDDING MODEL
# ============================================================================

print(f"\n[1] Loading embedding model: {MODEL_CONFIG.name} ({MODEL_CONFIG.hf_model_id})")

if MODEL_CONFIG.name in ("v2-moe", "v1.5"):
    # Sentence-transformers models
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(
        MODEL_CONFIG.hf_model_id,
        trust_remote_code=True,
    )
    model_backend = "sentence_transformers"
    print(f"  Backend: sentence-transformers")

elif MODEL_CONFIG.name == "bge-m3":
    # FlagEmbedding model
    from FlagEmbedding import BGEM3FlagModel
    embedding_model = BGEM3FlagModel(
        MODEL_CONFIG.hf_model_id,
        use_fp16=True,
    )
    model_backend = "flag_embedding"
    print(f"  Backend: FlagEmbedding (BGE-M3)")

else:
    raise ValueError(f"Unknown model: {MODEL_CONFIG.name}")

print(f"✓ Model loaded")

# ============================================================================
# ENCODE ENRICHED CHUNKS
# ============================================================================

print(f"\n[2] Encoding {len(enriched_chunks)} enriched chunks...")

chunk_texts = [c.text for c in enriched_chunks]
chunk_ids = [c.chunk_id for c in enriched_chunks]

if model_backend == "sentence_transformers":
    embeddings = embedding_model.encode(
        chunk_texts,
        prompt_name="document",
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
elif model_backend == "flag_embedding":
    result = embedding_model.encode(chunk_texts, batch_size=32, max_length=8192)
    embeddings = np.array(result["dense_vecs"])

print(f"✓ Encoded {len(embeddings)} chunks")
print(f"  Embedding dimensions: {embeddings.shape[1]}")

# ============================================================================
# CHROMADB COLLECTION SETUP
# ============================================================================

print(f"\n[3] Setting up ChromaDB collection...")

client = chromadb.PersistentClient(path=str(CHROMADB_DIR))
collection_name = f"d23_multi_layer_{MODEL_CONFIG.name}"

# Delete existing collection if present (clean slate)
try:
    client.delete_collection(name=collection_name)
    print(f"  Deleted existing collection: {collection_name}")
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"},
)
print(f"✓ Collection created: {collection_name}")

# ============================================================================
# INDEX CHUNKS
# ============================================================================

print(f"\n[4] Adding chunks to collection...")

# Prepare metadata for ChromaDB (flatten complex types to strings)
chroma_metadatas = []
for chunk in enriched_chunks:
    flat_meta = {}
    for key, value in chunk.metadata.items():
        if isinstance(value, (list, dict)):
            flat_meta[key] = json.dumps(value)  # Serialize complex types
        elif value is not None:
            flat_meta[key] = str(value)
    chroma_metadatas.append(flat_meta)

# Add in batches
batch_size = 100
for i in range(0, len(chunk_ids), batch_size):
    end = min(i + batch_size, len(chunk_ids))
    collection.add(
        ids=chunk_ids[i:end],
        embeddings=embeddings[i:end].tolist(),
        documents=chunk_texts[i:end],
        metadatas=chroma_metadatas[i:end],
    )

print(f"✓ Indexed {collection.count()} chunks")

# ============================================================================
# VERIFICATION
# ============================================================================

print(f"\n[5] Collection summary:")
print(f"  Collection name:    {collection_name}")
print(f"  Chunks indexed:     {collection.count()}")
print(f"  Embedding dims:     {embeddings.shape[1]}")
print(f"  Distance metric:    cosine")
print(f"  Model:              {MODEL_CONFIG.name}")
print(f"  Enrichment:         multi-layer (8 layers)")

print("\n✓ Embedding and indexing complete.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 12: Query Execution

Runs all 36 ground-truth queries against the D-23 ChromaDB collection.
Retrieves top-10 results per query with cosine distance scores.

Same pipeline as D-22 Cell 12 but querying the multi-layer collection.

Outputs:
  - query_results: List[QueryResult] (one per query, with top-10 results)
"""

@dataclass
class QueryResult:
    """Result of executing a single query against the D-23 collection.

    Attributes:
        query_id: Query identifier (e.g., "Q-01")
        query_text: Natural language query string
        query_type: Query category (SINGLE_HOP, MULTI_HOP, AUTHORITY, TEMPORAL, EXPLORATORY)
        results: List of result dicts, each with:
            - chunk_id (str): Matched chunk identifier
            - distance (float): Cosine distance (lower = more similar)
            - text_preview (str): First 200 chars of matched text
            - metadata (dict): Chunk metadata
    """
    query_id: str
    query_text: str
    query_type: str
    results: List[Dict[str, Any]]


def encode_query(query_text: str) -> np.ndarray:
    """Encode a query string into the embedding space.

    Uses the appropriate task prefix for the selected model.

    Args:
        query_text: Natural language query

    Returns:
        1D numpy array (embedding vector)
    """
    if model_backend == "sentence_transformers":
        return embedding_model.encode(
            [query_text],
            prompt_name="query",
            convert_to_numpy=True,
        )[0]
    elif model_backend == "flag_embedding":
        result = embedding_model.encode([query_text])
        return np.array(result["dense_vecs"][0])
    else:
        raise ValueError(f"Unknown backend: {model_backend}")


def run_query(query_text: str, top_k: int = 10) -> List[Dict[str, Any]]:
    """Execute a single query against the D-23 collection.

    Args:
        query_text: Query string
        top_k: Number of results to retrieve (default 10)

    Returns:
        List of result dicts with chunk_id, distance, text_preview, metadata
    """
    query_embedding = encode_query(query_text)

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )

    formatted = []
    if results["ids"] and results["ids"][0]:
        for i, (cid, dist, doc, meta) in enumerate(zip(
            results["ids"][0],
            results["distances"][0],
            results["documents"][0],
            results["metadatas"][0],
        )):
            formatted.append({
                "rank": i + 1,
                "chunk_id": cid,
                "distance": float(dist),
                "text_preview": doc[:200] + "..." if len(doc) > 200 else doc,
                "metadata": meta,
            })

    return formatted


# ============================================================================
# EXECUTE ALL QUERIES
# ============================================================================

print("=" * 80)
print("QUERY EXECUTION")
print("=" * 80)
print(f"\n  Running {len(GROUND_TRUTH_QUERIES)} queries against {collection_name}...")

query_results: List[QueryResult] = []

for i, gt_query in enumerate(tqdm(GROUND_TRUTH_QUERIES, desc="Querying")):
    results = run_query(gt_query.query_text, top_k=10)
    query_results.append(QueryResult(
        query_id=gt_query.query_id,
        query_text=gt_query.query_text,
        query_type=gt_query.query_type,
        results=results,
    ))

# ============================================================================
# SUMMARY
# ============================================================================

results_per_q = [len(qr.results) for qr in query_results]
print(f"\n✓ All {len(query_results)} queries executed")
print(f"  Avg results/query:  {np.mean(results_per_q):.1f}")
print(f"  Total result tuples: {sum(results_per_q)}")

# By query type
for qtype in ["SINGLE_HOP", "MULTI_HOP", "AUTHORITY", "TEMPORAL", "EXPLORATORY"]:
    count = sum(1 for qr in query_results if qr.query_type == qtype)
    print(f"  {qtype:12s}: {count} queries")

# Sample preview
print(f"\nSample results (first 2 queries):")
for qr in query_results[:2]:
    print(f"  {qr.query_id} [{qr.query_type}]: {qr.query_text[:50]}...")
    for res in qr.results[:3]:
        print(f"    [{res['rank']}] {res['chunk_id']} (dist={res['distance']:.4f})")

print("\n✓ Ready for metric computation.")


## D-23 Results: 3-Way Comparison Analysis

All 36 ground-truth queries have been executed against the D-23 multi-layer enriched corpus. The following cells compute retrieval metrics and perform a **3-way comparison** across:

1. **D-21 Baseline** — no enrichment (raw chunks)
2. **D-22 Single-Layer** — one-sentence document-level prefix (~15-25 tokens)
3. **D-23 Multi-Layer** — 8-layer composite prefix (~50-150 tokens per chunk)

### Statistical Framework

With 3 pairwise comparisons, **Bonferroni correction** controls the family-wise error rate:

- **Adjusted α** = 0.05 / 3 ≈ **0.0167**
- A test is significant only if **p < 0.0167**

This is conservative but appropriate for an early-stage decision point.

### What Follows

| Cell | Purpose |
|------|---------|
| 14 | Metric computation functions (Precision@5, Recall@10, NDCG@10, MRR) |
| 15 | Compute D-23 metrics; load D-21 and D-22 results; merge comparison dataframe |
| 16 | 3-way delta analysis: per-query deltas, query-type breakdown, marginal value |
| 17 | Wilcoxon signed-rank tests with Bonferroni correction; effect sizes |
| 18 | Visualization: 3-way comparison bar chart |
| 19 | Visualization: delta heatmap and layer token distribution |
| 20 | **GO/NO-GO Decision Engine**: automated evaluation of 7 criteria |
| 21 | Export all results, CSVs, decision file |
| 22 | Decision summary and next steps |

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 14: Metric Computation Functions

Defines the four retrieval evaluation metrics used across D-21, D-22, and D-23:
  1. Precision@K — fraction of top-K results that are relevant
  2. Recall@K — fraction of all relevant documents found in top-K
  3. NDCG@K — ranking quality using graded relevance with log discount
  4. MRR — reciprocal rank of first relevant result

Also defines compute_all_metrics() which runs all four on a list of QueryResults.

These functions are identical to D-21 Cell 14 and D-22 Cell 14.
"""


def precision_at_k(
    retrieved_ids: List[str],
    relevant_ids: set,
    k: int = K_PRECISION,
) -> float:
    """Compute Precision@K.

    Formula: P@K = |relevant ∩ top-K| / K

    Args:
        retrieved_ids: Ordered list of retrieved chunk IDs (rank order)
        relevant_ids: Set of known-relevant chunk IDs
        k: Cutoff rank (default K_PRECISION=5)

    Returns:
        Precision@K in [0.0, 1.0]

    Example:
        >>> precision_at_k(["A","B","C","D","E"], {"A","C","F"}, k=5)
        0.4  # 2 relevant in top-5
    """
    if k <= 0:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & set(relevant_ids)) / k


def recall_at_k(
    retrieved_ids: List[str],
    relevant_ids: set,
    k: int = K_RECALL,
) -> float:
    """Compute Recall@K.

    Formula: R@K = |relevant ∩ top-K| / |relevant|

    Args:
        retrieved_ids: Ordered list of retrieved chunk IDs
        relevant_ids: Set of known-relevant chunk IDs
        k: Cutoff rank (default K_RECALL=10)

    Returns:
        Recall@K in [0.0, 1.0]. Returns 0.0 if relevant set is empty.

    Example:
        >>> recall_at_k(["A","B","C"], {"A","C","D","E"}, k=3)
        0.5  # 2 of 4 relevant found
    """
    if not relevant_ids:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & set(relevant_ids)) / len(relevant_ids)


def ndcg_at_k(
    retrieved_ids: List[str],
    relevance_scores: Dict[str, int],
    k: int = K_NDCG,
) -> float:
    """Compute NDCG@K (Normalized Discounted Cumulative Gain).

    Formula:
      DCG@K  = Σ(i=1..K) rel(i) / log2(i + 1)
      IDCG@K = DCG of ideal ranking (sorted by relevance descending)
      NDCG@K = DCG@K / IDCG@K

    Uses graded relevance scores (1=marginal, 2=relevant, 3=highly relevant).

    IMPORTANT: Retrieved IDs are deduplicated before scoring. In chunk-level
    retrieval with document-level relevance, multiple chunks from the same
    document map to the same ID after .split("#")[0]. Without dedup, DCG
    accumulates gains for every duplicate while IDCG only counts unique docs,
    producing NDCG > 1.0 (mathematically impossible).
    [FIX: Round 3 — dedup retrieved IDs before DCG computation]

    Args:
        retrieved_ids: Ordered list of retrieved chunk/document IDs
        relevance_scores: Dict mapping doc_id → relevance grade
        k: Cutoff rank (default K_NDCG=10)

    Returns:
        NDCG@K in [0.0, 1.0]. Returns 0.0 if no relevant documents exist.

    Example:
        >>> ndcg_at_k(["A","B","C"], {"A": 3, "C": 1}, k=3)
        # DCG = 3/log2(2) + 0/log2(3) + 1/log2(4) = 3.0 + 0 + 0.5 = 3.5
        # IDCG = 3/log2(2) + 1/log2(3) = 3.0 + 0.63 = 3.63
        # NDCG = 3.5 / 3.63 ≈ 0.964
    """
    if not relevance_scores:
        return 0.0

    # Deduplicate: only keep first occurrence of each document ID.
    # This prevents counting the same document multiple times when
    # multiple chunks from one doc appear in the top-k results.
    seen = set()
    deduped = []
    for rid in retrieved_ids[:k]:
        if rid not in seen:
            seen.add(rid)
            deduped.append(rid)

    # Compute DCG on deduplicated list
    dcg = 0.0
    for i, chunk_id in enumerate(deduped):
        rel = relevance_scores.get(chunk_id, 0)
        dcg += rel / np.log2(i + 2)  # i+2 because 0-indexed

    # Compute ideal DCG (IDCG)
    ideal_rels = sorted(relevance_scores.values(), reverse=True)[:k]
    idcg = 0.0
    for i, rel in enumerate(ideal_rels):
        idcg += rel / np.log2(i + 2)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_reciprocal_rank(
    retrieved_ids: List[str],
    relevant_ids: set,
) -> float:
    """Compute Mean Reciprocal Rank (MRR).

    Formula: MRR = 1 / rank_of_first_relevant_result

    Args:
        retrieved_ids: Ordered list of retrieved chunk IDs
        relevant_ids: Set of known-relevant chunk IDs

    Returns:
        MRR in [0.0, 1.0]. Returns 0.0 if no relevant result found.

    Example:
        >>> mean_reciprocal_rank(["X","Y","A","B"], {"A","C"})
        0.333  # First relevant at rank 3
    """
    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def compute_all_metrics(
    qr_list: List[QueryResult],
    gt_queries: List[GroundTruthQuery],
) -> pd.DataFrame:
    """Compute all four metrics for each query.

    Matches QueryResult objects to GroundTruthQuery objects by query_id.
    Returns a DataFrame with one row per query.

    Args:
        qr_list: List of QueryResult objects from query execution
        gt_queries: List of GroundTruthQuery objects with relevance judgments

    Returns:
        DataFrame with columns:
          [query_id, query_type, precision@5, recall@10, ndcg@10, mrr]
    """
    # Build lookup: query_id → ground truth
    gt_lookup = {q.query_id: q for q in gt_queries}

    rows = []
    for qr in qr_list:
        gt = gt_lookup.get(qr.query_id)
        if gt is None:
            continue

        retrieved = [r["chunk_id"].split("#")[0] for r in qr.results]
        relevant = set(gt.expected_filenames)

        rows.append({
            "query_id": qr.query_id,
            "query_type": qr.query_type,
            "precision@5": precision_at_k(retrieved, relevant),
            "recall@10": recall_at_k(retrieved, relevant),
            "ndcg@10": ndcg_at_k(retrieved, gt.relevance_scores),
            "mrr": mean_reciprocal_rank(retrieved, relevant),
        })

    return pd.DataFrame(rows)


# ============================================================================
# VERIFICATION
# ============================================================================

print("✓ Metric Computation Functions Loaded")
print(f"  Functions: precision_at_k, recall_at_k, ndcg_at_k, mean_reciprocal_rank")
print(f"  compute_all_metrics → DataFrame[query_id, query_type, P@5, R@10, NDCG@10, MRR]")
print(f"  Evaluation constants: K_PRECISION={K_PRECISION}, K_RECALL={K_RECALL}, K_NDCG={K_NDCG}")
print("✓ Ready for metric computation.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 15: Compute D-23 Metrics & Load Prior Results

Computes per-query metrics for D-23 multi-layer enrichment, loads D-21
baseline and D-22 single-layer results, and builds a unified comparison
DataFrame for 3-way delta analysis.

Outputs:
  - d23_df: DataFrame with D-23 per-query metrics
  - d21_df: DataFrame with D-21 baseline metrics (loaded from CSV)
  - d22_df: DataFrame with D-22 single-layer metrics (loaded from CSV)
  - comparison_df: Combined DataFrame with experiment labels

Reference: D-21 Cell 15 (baseline export format), D-22 Cell 15 (comparison pattern)
"""

import warnings

# ============================================================================
# COMPUTE D-23 METRICS
# ============================================================================

print("=" * 90)
print("COMPUTING D-23 MULTI-LAYER ENRICHMENT METRICS")
print("=" * 90)

d23_df = compute_all_metrics(query_results, GROUND_TRUTH_QUERIES)
d23_df["model"] = SELECTED_MODEL
d23_df["experiment"] = "multi_layer"

print(f"\n✓ D-23 metrics computed for {len(d23_df)} queries")
print(f"  Columns: {list(d23_df.columns)}")
print(f"  Model: {SELECTED_MODEL}")
print(f"  Experiment: multi_layer")

# ============================================================================
# LOAD D-21 BASELINE RESULTS
# ============================================================================

print()
print("-" * 90)
print("Loading D-21 Baseline Results")
print("-" * 90)

if D21_RESULTS_PATH.exists():
    d21_df = pd.read_csv(D21_RESULTS_PATH)
    d21_df["experiment"] = "baseline"
    print(f"✓ Loaded D-21 baseline: {len(d21_df)} queries from {D21_RESULTS_PATH}")
else:
    warnings.warn(f"⚠ D-21 results not found at {D21_RESULTS_PATH}")
    print(f"⚠ D-21 results file not found: {D21_RESULTS_PATH}")
    print("  D-23 vs D-21 comparison will be skipped.")
    d21_df = pd.DataFrame()

# ============================================================================
# LOAD D-22 SINGLE-LAYER RESULTS
# ============================================================================

print()
print("-" * 90)
print("Loading D-22 Single-Layer Results")
print("-" * 90)

if D22_RESULTS_PATH.exists():
    d22_df = pd.read_csv(D22_RESULTS_PATH)
    d22_df["experiment"] = "single_layer"
    print(f"✓ Loaded D-22 single-layer: {len(d22_df)} queries from {D22_RESULTS_PATH}")
else:
    warnings.warn(f"⚠ D-22 results not found at {D22_RESULTS_PATH}")
    print(f"⚠ D-22 results file not found: {D22_RESULTS_PATH}")
    print("  D-23 vs D-22 comparison will be skipped.")
    d22_df = pd.DataFrame()

# ============================================================================
# BUILD COMPARISON DATAFRAME
# ============================================================================

print()
print("-" * 90)
print("Building 3-Way Comparison DataFrame")
print("-" * 90)

frames_to_concat = [d23_df]
if not d21_df.empty:
    frames_to_concat.append(d21_df)
if not d22_df.empty:
    frames_to_concat.append(d22_df)

comparison_df = pd.concat(frames_to_concat, ignore_index=True)
print(f"✓ Comparison DataFrame: {len(comparison_df)} total rows")
print(f"  Experiments present: {comparison_df['experiment'].unique().tolist()}")

# ============================================================================
# SUMMARY TABLE
# ============================================================================

print()
print("=" * 90)
print("MEAN METRIC VALUES BY EXPERIMENT")
print("=" * 90)

summary_stats = comparison_df.groupby("experiment")[METRICS].mean()
print()
print(summary_stats.to_string())

# ============================================================================
# TREND ARROWS (D-23 vs D-22 and D-22 vs D-21)
# ============================================================================

print()
print("=" * 90)
print("TREND ANALYSIS")
print("=" * 90)

d23_means = d23_df[METRICS].mean()

if not d22_df.empty:
    d22_means = d22_df[METRICS].mean()
    print("\nD-23 (Multi-Layer) vs D-22 (Single-Layer):")
    for metric in METRICS:
        d23_val = d23_means[metric]
        d22_val = d22_means[metric]
        delta = d23_val - d22_val

        if abs(delta) < 0.001:
            arrow = "→"
        elif delta > 0:
            arrow = "↑"
        else:
            arrow = "↓"

        pct = (delta / d22_val * 100) if d22_val > 0 else 0
        label = METRIC_LABELS[METRICS.index(metric)]
        print(f"  {label:15} {arrow} {d23_val:.4f} vs {d22_val:.4f} (delta {delta:+.4f}, {pct:+.1f}%)")

if not d21_df.empty:
    d21_means = d21_df[METRICS].mean()
    print("\nD-23 (Multi-Layer) vs D-21 (Baseline):")
    for metric in METRICS:
        d23_val = d23_means[metric]
        d21_val = d21_means[metric]
        delta = d23_val - d21_val

        if abs(delta) < 0.001:
            arrow = "→"
        elif delta > 0:
            arrow = "↑"
        else:
            arrow = "↓"

        pct = (delta / d21_val * 100) if d21_val > 0 else 0
        label = METRIC_LABELS[METRICS.index(metric)]
        print(f"  {label:15} {arrow} {d23_val:.4f} vs {d21_val:.4f} (delta {delta:+.4f}, {pct:+.1f}%)")

print()
print("=" * 90)
print(f"✓ D-23 metrics ready. Total queries: {len(d23_df)}")
print("✓ Comparison DataFrame built. Ready for 3-way delta analysis.")
print("=" * 90)


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 16: 3-Way Delta Analysis — THE CORE ANALYSIS CELL

Computes per-query metric deltas for all 3 comparison pairs:
  1. ml_vs_bl: D-23 (multi-layer) - D-21 (baseline) — primary hypothesis
  2. ml_vs_sl: D-23 (multi-layer) - D-22 (single-layer) — complexity justification
  3. sl_vs_bl: D-22 (single-layer) - D-21 (baseline) — reference / sanity check

Also computes MARGINAL VALUE: the improvement added by layers 2-8 beyond
the single-layer prefix (isolates the contribution of the additional layers).

Outputs:
  - delta_results: dict of pair_key → delta DataFrame
  - d23_delta_vs_d21.csv, d23_delta_vs_d22.csv

Reference: D-22 Cell 16 (2-way delta pattern); D-23 Cell 07 (methodology)
"""

print("=" * 90)
print("3-WAY DELTA ANALYSIS: MULTI-LAYER vs SINGLE-LAYER vs BASELINE")
print("=" * 90)

# ============================================================================
# DEFINE COMPARISON PAIRS
# ============================================================================

PAIRS = [
    ("multi_layer_vs_baseline",  "D-23 MULTI-LAYER vs D-21 BASELINE",  "multi_layer", "baseline"),
    ("multi_layer_vs_single",    "D-23 MULTI-LAYER vs D-22 SINGLE-LAYER", "multi_layer", "single_layer"),
    ("single_vs_baseline",       "D-22 SINGLE-LAYER vs D-21 BASELINE",  "single_layer", "baseline"),
]

delta_results = {}  # pair_key → DataFrame

# ============================================================================
# COMPUTE DELTAS FOR EACH PAIR
# ============================================================================

for pair_key, pair_label, exp_a, exp_b in PAIRS:
    print(f"\n{'=' * 90}")
    print(f"{pair_label}")
    print(f"{'=' * 90}")

    # Extract data for each experiment
    df_a = comparison_df[comparison_df["experiment"] == exp_a].copy()
    df_b = comparison_df[comparison_df["experiment"] == exp_b].copy()

    if df_a.empty or df_b.empty:
        print(f"  ⚠ Missing data for this pair. Skipping.")
        continue

    # Merge on query_id for paired comparison
    merged = pd.merge(
        df_a, df_b,
        on="query_id",
        suffixes=("_a", "_b"),
        how="inner",
    )

    if merged.empty:
        print(f"  ⚠ No matching queries found. Skipping.")
        continue

    # Build delta DataFrame
    delta_rows = []
    for _, row in merged.iterrows():
        delta_p5   = row["precision@5_a"]  - row["precision@5_b"]
        delta_r10  = row["recall@10_a"]    - row["recall@10_b"]
        delta_ndcg = row["ndcg@10_a"]      - row["ndcg@10_b"]
        delta_mrr  = row["mrr_a"]          - row["mrr_b"]

        # Classification flags
        improved = (delta_p5 > 0) or (delta_r10 > 0) or (delta_ndcg > 0) or (delta_mrr > 0)
        degraded = (delta_p5 < 0) or (delta_r10 < 0) or (delta_ndcg < 0) or (delta_mrr < 0)

        delta_rows.append({
            "query_id": row["query_id"],
            "query_type": row["query_type_a"],
            "delta_p5": delta_p5,
            "delta_r10": delta_r10,
            "delta_ndcg": delta_ndcg,
            "delta_mrr": delta_mrr,
            "improved": improved,
            "degraded": degraded,
            "all_metrics_improved": all([delta_p5 >= 0, delta_r10 >= 0, delta_ndcg >= 0, delta_mrr >= 0]),
        })

    delta_df = pd.DataFrame(delta_rows)
    delta_results[pair_key] = delta_df

    # ---- OVERALL STATISTICS ----
    print(f"\n  OVERALL STATISTICS ({len(delta_df)} queries)")
    print(f"  {'-' * 80}")

    metrics_delta = ["delta_p5", "delta_r10", "delta_ndcg", "delta_mrr"]

    for metric_d, label in zip(metrics_delta, METRIC_LABELS):
        mean_d  = delta_df[metric_d].mean()
        median_d = delta_df[metric_d].median()
        std_d   = delta_df[metric_d].std()
        improved_n = (delta_df[metric_d] > 0).sum()
        degraded_n = (delta_df[metric_d] < 0).sum()
        unchanged_n = (delta_df[metric_d] == 0).sum()

        print(f"\n  {label}:")
        print(f"    Mean delta:  {mean_d:+.4f}")
        print(f"    Median delta: {median_d:+.4f}")
        print(f"    Std dev:     {std_d:.4f}")
        print(f"    Improved: {improved_n} ({improved_n/len(delta_df)*100:.1f}%)")
        print(f"    Degraded: {degraded_n} ({degraded_n/len(delta_df)*100:.1f}%)")
        print(f"    Unchanged: {unchanged_n} ({unchanged_n/len(delta_df)*100:.1f}%)")

    # ---- BY QUERY TYPE ----
    print(f"\n  BY QUERY TYPE")
    print(f"  {'-' * 80}")

    for query_type in ["SINGLE_HOP", "MULTI_HOP", "AUTHORITY", "TEMPORAL", "EXPLORATORY"]:
        type_data = delta_df[delta_df["query_type"] == query_type]
        if type_data.empty:
            continue

        print(f"\n  {query_type.upper()} ({len(type_data)} queries)")
        for metric_d, label in zip(metrics_delta, METRIC_LABELS):
            mean_d = type_data[metric_d].mean()
            improved_n = (type_data[metric_d] > 0).sum()
            print(f"    {label:15} mean_delta={mean_d:+.4f}, improved={improved_n}/{len(type_data)}")

    # ---- QUERY CLASSIFICATION ----
    print(f"\n  QUERY CLASSIFICATION")
    print(f"    Any metric improved:    {delta_df['improved'].sum()} ({delta_df['improved'].sum()/len(delta_df)*100:.1f}%)")
    print(f"    Any metric degraded:    {delta_df['degraded'].sum()} ({delta_df['degraded'].sum()/len(delta_df)*100:.1f}%)")
    print(f"    All metrics improved:   {delta_df['all_metrics_improved'].sum()} ({delta_df['all_metrics_improved'].sum()/len(delta_df)*100:.1f}%)")

# ============================================================================
# MARGINAL VALUE ANALYSIS
# ============================================================================

print(f"\n\n{'=' * 90}")
print("MARGINAL VALUE ANALYSIS: Value Added by Layers 2-8")
print("=" * 90)
print()
print("Formula: (D-23 improvement over D-21) - (D-22 improvement over D-21)")
print("         = value added by the additional 7 layers beyond single-layer prefix")

if "multi_layer_vs_baseline" in delta_results and "single_vs_baseline" in delta_results:
    ml_bl = delta_results["multi_layer_vs_baseline"]
    sl_bl = delta_results["single_vs_baseline"]

    # Merge on query_id
    marginal_df = pd.merge(
        ml_bl[["query_id"] + metrics_delta],
        sl_bl[["query_id"] + metrics_delta],
        on="query_id",
        suffixes=("_ml_bl", "_sl_bl"),
    )

    print()
    for metric_d, label in zip(metrics_delta, METRIC_LABELS):
        ml_improvement = marginal_df[f"{metric_d}_ml_bl"].mean()
        sl_improvement = marginal_df[f"{metric_d}_sl_bl"].mean()
        marginal_value = ml_improvement - sl_improvement
        pct_positive = (marginal_df[f"{metric_d}_ml_bl"] > marginal_df[f"{metric_d}_sl_bl"]).sum()

        trend = "↑" if marginal_value > 0 else ("↓" if marginal_value < 0 else "→")
        print(f"  {label:15} {trend} marginal={marginal_value:+.4f} (ML improvement={ml_improvement:+.4f}, SL improvement={sl_improvement:+.4f})")
        print(f"                    positive marginal in {pct_positive}/{len(marginal_df)} queries")
else:
    print("  ⚠ Cannot compute marginal value: need both D-21 and D-22 data.")

# ============================================================================
# EXPORT DELTA CSVS
# ============================================================================

print(f"\n{'=' * 90}")
print("EXPORTING DELTA RESULTS")
print("=" * 90)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "multi_layer_vs_baseline" in delta_results:
    path = OUTPUT_DIR / "d23_delta_vs_d21.csv"
    delta_results["multi_layer_vs_baseline"].to_csv(path, index=False)
    print(f"✓ Exported: {path} ({len(delta_results['multi_layer_vs_baseline'])} queries)")

if "multi_layer_vs_single" in delta_results:
    path = OUTPUT_DIR / "d23_delta_vs_d22.csv"
    delta_results["multi_layer_vs_single"].to_csv(path, index=False)
    print(f"✓ Exported: {path} ({len(delta_results['multi_layer_vs_single'])} queries)")

print()
print("✓ Delta analysis complete. Ready for statistical significance testing.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 17: Statistical Significance Testing — 3-Way with Bonferroni

Performs Wilcoxon signed-rank test for all 12 combinations (3 pairs × 4 metrics).

Key differences from D-22:
  - 3 comparison pairs instead of 1
  - Bonferroni correction: adjusted α = 0.05 / 3 ≈ 0.0167
  - Rank-biserial correlation as effect size measure

Effect size interpretation (rank-biserial correlation):
  - |r| < 0.1:  negligible
  - 0.1-0.3:    small
  - 0.3-0.5:    medium
  - > 0.5:      large

Reference: scipy.stats.wilcoxon; Bonferroni correction; D-22 Cell 17
"""

from scipy.stats import wilcoxon

print("=" * 90)
print("STATISTICAL SIGNIFICANCE TESTING (BONFERRONI-CORRECTED)")
print("=" * 90)
print()
print(f"Test: Wilcoxon signed-rank (paired, non-parametric)")
print(f"Sample: 36 paired queries per comparison")
print(f"Bonferroni correction: {BONFERRONI_PAIRS} pairs")
print(f"  Unadjusted α = {SIGNIFICANCE_LEVEL}")
print(f"  Adjusted α   = {ADJUSTED_ALPHA:.4f}")
print(f"Total tests: {BONFERRONI_PAIRS} pairs × {len(METRICS)} metrics = {BONFERRONI_PAIRS * len(METRICS)}")

# ============================================================================
# RUN WILCOXON TESTS FOR ALL 12 COMBINATIONS
# ============================================================================

significance_results = []

for pair_key, pair_label, exp_a, exp_b in PAIRS:
    if pair_key not in delta_results:
        continue

    delta_df = delta_results[pair_key]

    print(f"\n{'-' * 90}")
    print(f"{pair_label}")
    print(f"{'-' * 90}")

    metrics_delta = ["delta_p5", "delta_r10", "delta_ndcg", "delta_mrr"]

    for metric_d, label in zip(metrics_delta, METRIC_LABELS):
        deltas = delta_df[metric_d].values

        # Filter non-zero deltas (Wilcoxon requires non-zero differences)
        nonzero = deltas[deltas != 0]

        if len(nonzero) < 2:
            print(f"  {label:15} — insufficient non-zero deltas ({len(nonzero)}); skipping")
            significance_results.append({
                "pair": pair_key,
                "pair_label": pair_label,
                "metric": label,
                "statistic": None,
                "pvalue": None,
                "significant": False,
                "effect_size": None,
                "effect_magnitude": "insufficient_data",
                "n_nonzero": len(nonzero),
            })
            continue

        # Wilcoxon signed-rank test (two-tailed)
        stat, pvalue = wilcoxon(nonzero)

        # Rank-biserial correlation as effect size
        # Formula: r = 1 - (2 * W) / (n * (n + 1) / 2)
        # where W is the Wilcoxon statistic, n is number of non-zero pairs
        n = len(nonzero)
        r_rb = 1 - (2 * stat) / (n * (n + 1) / 2)

        # Effect magnitude classification
        abs_r = abs(r_rb)
        if abs_r < 0.1:
            effect_mag = "negligible"
        elif abs_r < 0.3:
            effect_mag = "small"
        elif abs_r < 0.5:
            effect_mag = "medium"
        else:
            effect_mag = "large"

        # Bonferroni-corrected significance
        is_significant = pvalue < ADJUSTED_ALPHA
        sig_marker = "***" if is_significant else ""

        print(f"  {label:15} W={stat:8.1f}, p={pvalue:.6f}, r={r_rb:+.3f} ({effect_mag:10}) {sig_marker}")

        significance_results.append({
            "pair": pair_key,
            "pair_label": pair_label,
            "metric": label,
            "statistic": stat,
            "pvalue": pvalue,
            "significant": is_significant,
            "effect_size": r_rb,
            "effect_magnitude": effect_mag,
            "n_nonzero": n,
        })

# ============================================================================
# SIGNIFICANCE SUMMARY
# ============================================================================

sig_df = pd.DataFrame(significance_results)

print(f"\n{'=' * 90}")
print("SIGNIFICANCE SUMMARY")
print(f"{'=' * 90}")

total_tests = len(sig_df[sig_df["statistic"].notna()])
significant_tests = sig_df["significant"].sum()

print(f"\n  Total valid tests:     {total_tests}")
print(f"  Significant (p < {ADJUSTED_ALPHA:.4f}): {significant_tests} ({significant_tests/total_tests*100:.1f}%)" if total_tests > 0 else "  No valid tests")

print(f"\n  By Comparison Pair:")
for pair_key, pair_label, _, _ in PAIRS:
    pair_data = sig_df[sig_df["pair"] == pair_key]
    if not pair_data.empty:
        pair_sig = pair_data["significant"].sum()
        pair_total = len(pair_data[pair_data["statistic"].notna()])
        print(f"    {pair_label:50} {pair_sig}/{pair_total} significant")

print(f"\n  By Metric:")
for label in METRIC_LABELS:
    metric_data = sig_df[sig_df["metric"] == label]
    if not metric_data.empty:
        metric_sig = metric_data["significant"].sum()
        metric_total = len(metric_data[metric_data["statistic"].notna()])
        print(f"    {label:20} {metric_sig}/{metric_total} significant")

# Key finding for GO/NO-GO criterion 4
ml_bl_sig = sig_df[(sig_df["pair"] == "multi_layer_vs_baseline") & (sig_df["significant"])]
print(f"\n  CRITICAL (GO/NO-GO Criterion 4):")
print(f"    Multi-Layer vs Baseline: {len(ml_bl_sig)} significant metrics (threshold: ≥2)")

# ============================================================================
# EXPORT
# ============================================================================

sig_export = sig_df[["pair_label", "metric", "statistic", "pvalue", "significant",
                      "effect_size", "effect_magnitude", "n_nonzero"]].copy()
sig_path = OUTPUT_DIR / "d23_significance_results.csv"
sig_export.to_csv(sig_path, index=False)
print(f"\n✓ Exported: {sig_path}")
print("✓ Significance testing complete.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 18: Visualization — 3-Way Comparison Bar Chart

Creates a 2×2 figure:
  - Top-left: Overall performance (all 36 queries) — D-21 vs D-22 vs D-23
  - Top-right, Bottom-left, Bottom-right: Per-query-type comparison

Color scheme:
  - D-21 (Baseline):     #4e79a7 (steel blue)
  - D-22 (Single-Layer): #f28e2b (orange)
  - D-23 (Multi-Layer):  #59a14f (green)

Reference: D-22 Cell 18 (2-way visualization pattern)
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("=" * 90)
print("CREATING 3-WAY COMPARISON VISUALIZATION")
print("=" * 90)

# Color scheme
COLORS = {
    "baseline":     "#4e79a7",  # steel blue
    "single_layer": "#f28e2b",  # orange
    "multi_layer":  "#59a14f",  # green
}
EXP_LABELS = {
    "baseline":     "D-21 Baseline",
    "single_layer": "D-22 Single-Layer",
    "multi_layer":  "D-23 Multi-Layer",
}
experiments = ["baseline", "single_layer", "multi_layer"]

# Filter to only experiments present in comparison_df
available_exps = [e for e in experiments if e in comparison_df["experiment"].values]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("D-23: 3-Way Retrieval Performance Comparison",
             fontsize=16, fontweight="bold", y=0.98)

# ---- TOP-LEFT: Overall comparison (all queries) ----
ax = axes[0, 0]
x = np.arange(len(METRICS))
width = 0.25

for i, exp in enumerate(available_exps):
    exp_data = comparison_df[comparison_df["experiment"] == exp][METRICS].mean()
    offset = width * (i - len(available_exps) / 2 + 0.5)
    ax.bar(x + offset, exp_data.values, width,
           label=EXP_LABELS.get(exp, exp), color=COLORS.get(exp, "#999999"), alpha=0.85)

ax.set_xlabel("Metric", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Value", fontsize=11, fontweight="bold")
ax.set_title("Overall (All 36 Queries)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(METRIC_LABELS, fontsize=10)
ax.set_ylim(0, 1)
ax.legend(fontsize=9, loc="upper right")
ax.grid(axis="y", alpha=0.3)

# ---- REMAINING SUBPLOTS: Per query type ----
query_types = ["SINGLE_HOP", "MULTI_HOP", "AUTHORITY", "TEMPORAL", "EXPLORATORY"]
subplot_positions = [(0, 1), (1, 0), (1, 1)]

for qtype, (row, col) in zip(query_types, subplot_positions):
    ax = axes[row, col]
    type_df = comparison_df[comparison_df["query_type"] == qtype]
    x = np.arange(len(METRICS))

    for i, exp in enumerate(available_exps):
        exp_data = type_df[type_df["experiment"] == exp][METRICS].mean()
        if exp_data.empty:
            continue
        offset = width * (i - len(available_exps) / 2 + 0.5)
        ax.bar(x + offset, exp_data.values, width,
               label=EXP_LABELS.get(exp, exp), color=COLORS.get(exp, "#999999"), alpha=0.85)

    n_queries = len(type_df[type_df["experiment"] == "multi_layer"])
    ax.set_xlabel("Metric", fontsize=11, fontweight="bold")
    ax.set_ylabel("Mean Value", fontsize=11, fontweight="bold")
    ax.set_title(f"{qtype.capitalize()} Queries (n={n_queries})", fontsize=13, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(METRIC_LABELS, fontsize=10)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.3)

# Save
plt.tight_layout(rect=[0, 0, 1, 0.96])
viz_path = OUTPUT_DIR / "visualization_3way_comparison.png"
plt.savefig(viz_path, dpi=150, bbox_inches="tight")
print(f"\n✓ Saved: {viz_path}")
plt.close()

# Print key findings
print("\nKey Findings:")
for exp in available_exps:
    means = comparison_df[comparison_df["experiment"] == exp][METRICS].mean()
    overall_mean = means.mean()
    print(f"  {EXP_LABELS.get(exp, exp):25} overall mean: {overall_mean:.4f}")

print("\n✓ 3-way comparison visualization complete.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 19: Delta Heatmap & Layer Token Distribution

Figure 1 — Delta Heatmap:
  Two side-by-side heatmaps showing per-query metric deltas.
  Left:  D-23 vs D-21 (multi-layer vs baseline)
  Right: D-23 vs D-22 (multi-layer vs single-layer)
  Colormap: RdYlGn (diverging, center=0; red=degradation, green=improvement)

Figure 2 — Layer Token Distribution:
  Box plot showing token consumption per layer across all chunks.
  Informs D-24 ablation planning: high-token layers are candidates for removal.

Reference: D-22 Cell 19 (heatmap pattern); Cell 10 (layer_token_audit)
"""

import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 90)
print("CREATING DELTA HEATMAP & LAYER TOKEN DISTRIBUTION")
print("=" * 90)

# ============================================================================
# FIGURE 1: DELTA HEATMAPS
# ============================================================================

delta_cols = ["delta_p5", "delta_r10", "delta_ndcg", "delta_mrr"]
col_labels = ["ΔP@5", "ΔR@10", "ΔNDCG@10", "ΔMRR"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 14))
fig.suptitle("D-23: Per-Query Delta Heatmaps", fontsize=16, fontweight="bold")

# ---- Left: D-23 vs D-21 ----
if "multi_layer_vs_baseline" in delta_results:
    hm_df = delta_results["multi_layer_vs_baseline"].sort_values(
        by=["query_type", "query_id"]
    ).reset_index(drop=True)

    hm_data = hm_df[delta_cols].values
    query_labels = hm_df["query_id"].tolist()

    sns.heatmap(
        hm_data,
        annot=True, fmt=".3f", cmap="RdYlGn",
        center=0, vmin=-0.3, vmax=0.3,
        ax=ax1,
        cbar_kws={"label": "Delta"},
        xticklabels=col_labels,
        yticklabels=query_labels,
        annot_kws={"fontsize": 7},
    )
    ax1.set_title("D-23 vs D-21 (Multi-Layer vs Baseline)", fontsize=13, fontweight="bold")
    ax1.set_ylabel("Query ID", fontsize=11, fontweight="bold")
    ax1.set_xlabel("Metric Delta", fontsize=11, fontweight="bold")

# ---- Right: D-23 vs D-22 ----
if "multi_layer_vs_single" in delta_results:
    hm_df2 = delta_results["multi_layer_vs_single"].sort_values(
        by=["query_type", "query_id"]
    ).reset_index(drop=True)

    hm_data2 = hm_df2[delta_cols].values
    query_labels2 = hm_df2["query_id"].tolist()

    sns.heatmap(
        hm_data2,
        annot=True, fmt=".3f", cmap="RdYlGn",
        center=0, vmin=-0.3, vmax=0.3,
        ax=ax2,
        cbar_kws={"label": "Delta"},
        xticklabels=col_labels,
        yticklabels=query_labels2,
        annot_kws={"fontsize": 7},
    )
    ax2.set_title("D-23 vs D-22 (Multi-Layer vs Single-Layer)", fontsize=13, fontweight="bold")
    ax2.set_ylabel("Query ID", fontsize=11, fontweight="bold")
    ax2.set_xlabel("Metric Delta", fontsize=11, fontweight="bold")

plt.tight_layout(rect=[0, 0, 1, 0.96])
heatmap_path = OUTPUT_DIR / "visualization_delta_heatmap.png"
plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
print(f"✓ Saved: {heatmap_path}")
plt.close()

# ============================================================================
# FIGURE 2: LAYER TOKEN DISTRIBUTION
# ============================================================================

if layer_token_audits:
    fig, ax = plt.subplots(figsize=(14, 7))

    # Extract per-layer token data from audit
    layer_names = ["Corpus", "Domain", "Entity", "Authority",
                   "Temporal", "Relationships", "Section"]
    layer_data = {name: [] for name in layer_names}

    for entry in layer_token_audits:
        for name in layer_names:
            if name in entry and entry[name] > 0:
                layer_data[name].append(entry[name])

    # Filter to layers with data
    plot_layers = [n for n in layer_names if layer_data[n]]
    plot_data = [layer_data[n] for n in plot_layers]

    # Check if plot_data is empty before calling boxplot
    if not plot_data:
        print("⚠ No data available for layer token distribution plot; skipping.")
    else:
        bp = ax.boxplot(plot_data, labels=[n for n in plot_layers],
                         patch_artist=True, showfliers=True)

        # Color boxes by median token count (gradient: light=few, dark=many)
        medians = [np.median(d) for d in plot_data]
        max_median = max(medians) if medians else 1
        for patch, median_val in zip(bp["boxes"], medians):
            intensity = 0.3 + 0.7 * (median_val / max_median)
            patch.set_facecolor((0.35, 0.63, 0.31, intensity))  # green gradient
            patch.set_edgecolor("black")

        ax.set_ylabel("Tokens per Layer", fontsize=12, fontweight="bold")
        ax.set_xlabel("Enrichment Layer", fontsize=12, fontweight="bold")
        ax.set_title("Token Distribution Across Enrichment Layers\n(Informs D-24 Ablation Priority)",
                     fontsize=14, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        token_path = OUTPUT_DIR / "visualization_layer_token_distribution.png"
        plt.savefig(token_path, dpi=150, bbox_inches="tight")
        print(f"✓ Saved: {token_path}")
        plt.close()

        # Print summary
        print(f"\nLayer Token Usage Summary (for D-24 ablation planning):")
        for name in plot_layers:
            data = layer_data[name]
            print(f"  {name:15} median={np.median(data):5.1f}, mean={np.mean(data):5.1f}, max={np.max(data):5.1f}, present={len(data)}/{len(layer_token_audits)}")

else:
    print("⚠ layer_token_audits not available; skipping token distribution.")

print("\n✓ Delta heatmap and layer token distribution complete.")

In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 20: GO/NO-GO Decision Engine — THE DECISION CELL

Evaluates 7 success criteria programmatically:
  1. EXECUTION: < 5% token overflow
  2. IMPROVEMENT OVER D-21: mean delta > 0 for ≥3/4 metrics
  3. IMPROVEMENT OVER D-22: mean delta > 0 for ≥2/4 metrics
  4. STATISTICAL SIGNIFICANCE: Wilcoxon p < ADJUSTED_ALPHA for ≥2 metrics (ML vs BL)
  5. MARGINAL VALUE: ML vs SL improvement > 5% for ≥1 metric
  6. NO CATASTROPHIC DEGRADATION: < 25% queries degraded (ML vs BL)
  7. AUTHORITY/TEMPORAL BENEFIT: specialized queries show larger deltas than factual

Scoring:
  - ≥5/7 (AND criteria 2 + 4 both pass) → GO
  - 3-4/7 → CONDITIONAL GO
  - < 3/7 → NO-GO

Outputs:
  - go_nogo_decision: str ("GO", "CONDITIONAL_GO", or "NO_GO")
  - decision_report: str (full evidence trail)

Reference: D-23 Cell 07 (success criteria); Cell 16-17 (delta + significance data)
"""

print("=" * 90)
print("GO/NO-GO DECISION ENGINE: 7-CRITERION EVALUATION")
print("=" * 90)

# Initialize
criteria_results = []  # List of (number, name, passed, detail_string)
decision_report = []
decision_report.append("D-23 GO/NO-GO DECISION REPORT")
decision_report.append("=" * 90)
decision_report.append(f"Model: {SELECTED_MODEL}")
decision_report.append(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
decision_report.append(f"Bonferroni-adjusted α: {ADJUSTED_ALPHA:.4f}")
decision_report.append("")

# ============================================================================
# CRITERION 1: EXECUTION (< 5% token overflow)
# ============================================================================

print("\n[1/7] EXECUTION — Token Overflow Rate")
print("-" * 80)

if "overflow_count" in dir() or "overflow_count" in globals():
    overflow_pct = (overflow_count / len(enriched_chunks) * 100) if len(enriched_chunks) > 0 else 0
    c1_pass = overflow_pct < 5
    c1_detail = f"Overflow: {overflow_count}/{len(enriched_chunks)} chunks ({overflow_pct:.2f}%). Threshold: <5%."
else:
    # Fallback: assume pass if overflow tracking wasn't available
    c1_pass = True
    c1_detail = "Overflow data not tracked; default PASS."
    overflow_pct = 0

print(f"  {c1_detail}")
print(f"  Result: {'PASS ✓' if c1_pass else 'FAIL ✗'}")
criteria_results.append((1, "EXECUTION", c1_pass, c1_detail))
decision_report.append(f"[1] EXECUTION: {'PASS' if c1_pass else 'FAIL'} — {c1_detail}")

# ============================================================================
# CRITERION 2: IMPROVEMENT OVER D-21 (mean delta > 0 for ≥3/4 metrics)
# ============================================================================

print("\n[2/7] IMPROVEMENT OVER D-21 — Mean Delta Direction")
print("-" * 80)

if "multi_layer_vs_baseline" in delta_results:
    ml_bl = delta_results["multi_layer_vs_baseline"]
    c2_improved = 0
    for metric_d, label in zip(["delta_p5", "delta_r10", "delta_ndcg", "delta_mrr"], METRIC_LABELS):
        mean_d = ml_bl[metric_d].mean()
        is_up = mean_d > 0
        if is_up:
            c2_improved += 1
        print(f"  {label:15} mean_delta={mean_d:+.4f} {'✓' if is_up else '✗'}")

    c2_pass = c2_improved >= 3
    c2_detail = f"{c2_improved}/4 metrics have positive mean delta. Threshold: ≥3."
else:
    c2_pass = False
    c2_improved = 0
    c2_detail = "D-21 data unavailable."

print(f"\n  {c2_detail}")
print(f"  Result: {'PASS ✓' if c2_pass else 'FAIL ✗'}")
criteria_results.append((2, "IMPROVEMENT_OVER_D21", c2_pass, c2_detail))
decision_report.append(f"[2] IMPROVEMENT OVER D-21: {'PASS' if c2_pass else 'FAIL'} — {c2_detail}")

# ============================================================================
# CRITERION 3: IMPROVEMENT OVER D-22 (mean delta > 0 for ≥2/4 metrics)
# ============================================================================

print("\n[3/7] IMPROVEMENT OVER D-22 — Mean Delta Direction")
print("-" * 80)

if "multi_layer_vs_single" in delta_results:
    ml_sl = delta_results["multi_layer_vs_single"]
    c3_improved = 0
    for metric_d, label in zip(["delta_p5", "delta_r10", "delta_ndcg", "delta_mrr"], METRIC_LABELS):
        mean_d = ml_sl[metric_d].mean()
        is_up = mean_d > 0
        if is_up:
            c3_improved += 1
        print(f"  {label:15} mean_delta={mean_d:+.4f} {'✓' if is_up else '✗'}")

    c3_pass = c3_improved >= 2
    c3_detail = f"{c3_improved}/4 metrics have positive mean delta. Threshold: ≥2."
else:
    c3_pass = False
    c3_improved = 0
    c3_detail = "D-22 data unavailable."

print(f"\n  {c3_detail}")
print(f"  Result: {'PASS ✓' if c3_pass else 'FAIL ✗'}")
criteria_results.append((3, "IMPROVEMENT_OVER_D22", c3_pass, c3_detail))
decision_report.append(f"[3] IMPROVEMENT OVER D-22: {'PASS' if c3_pass else 'FAIL'} — {c3_detail}")

# ============================================================================
# CRITERION 4: STATISTICAL SIGNIFICANCE (≥2 metrics significant for ML vs BL)
# ============================================================================

print("\n[4/7] STATISTICAL SIGNIFICANCE — Wilcoxon (Bonferroni-Corrected)")
print("-" * 80)

if len(sig_df) > 0:
    ml_bl_sig = sig_df[(sig_df["pair"] == "multi_layer_vs_baseline") & sig_df["significant"]]
    c4_sig_count = len(ml_bl_sig)
    c4_pass = c4_sig_count >= 2
    c4_detail = f"{c4_sig_count}/4 metrics significant at α={ADJUSTED_ALPHA:.4f} for ML vs BL. Threshold: ≥2."

    # Show all ML vs BL results
    ml_bl_all = sig_df[sig_df["pair"] == "multi_layer_vs_baseline"]
    for _, row in ml_bl_all.iterrows():
        if row["statistic"] is not None:
            print(f"  {row['metric']:15} p={row['pvalue']:.6f}, r={row['effect_size']:+.3f} ({row['effect_magnitude']}) {'✓' if row['significant'] else '✗'}")
else:
    c4_pass = False
    c4_sig_count = 0
    c4_detail = "Significance data unavailable."

print(f"\n  {c4_detail}")
print(f"  Result: {'PASS ✓' if c4_pass else 'FAIL ✗'}")
criteria_results.append((4, "STATISTICAL_SIGNIFICANCE", c4_pass, c4_detail))
decision_report.append(f"[4] STATISTICAL SIGNIFICANCE: {'PASS' if c4_pass else 'FAIL'} — {c4_detail}")

# ============================================================================
# CRITERION 5: MARGINAL VALUE (ML vs SL gain > 5% for ≥1 metric)
# ============================================================================

print("\n[5/7] MARGINAL VALUE — Multi-Layer vs Single-Layer Percentage Gain")
print("-" * 80)

if "multi_layer_vs_single" in delta_results:
    ml_sl = delta_results["multi_layer_vs_single"]
    c5_above = 0
    for metric, label in zip(METRICS, METRIC_LABELS):
        metric_d = f"delta_{metric.replace('@', '')}" if "@" in metric else f"delta_{metric}"
        # Map metric name to delta column
        delta_col_map = {
            "precision@5": "delta_p5",
            "recall@10": "delta_r10",
            "ndcg@10": "delta_ndcg",
            "mrr": "delta_mrr",
        }
        delta_col = delta_col_map[metric]
        mean_delta = ml_sl[delta_col].mean()
        baseline_mean = comparison_df[comparison_df["experiment"] == "single_layer"][metric].mean()
        pct_gain = (mean_delta / baseline_mean * 100) if baseline_mean > 0 else 0

        above_5 = pct_gain > 5
        if above_5:
            c5_above += 1
        print(f"  {label:15} gain={pct_gain:+.2f}% {'✓' if above_5 else '✗'}")

    c5_pass = c5_above >= 1
    c5_detail = f"{c5_above}/4 metrics show >5% gain over single-layer. Threshold: ≥1."
else:
    c5_pass = False
    c5_above = 0
    c5_detail = "D-22 data unavailable."

print(f"\n  {c5_detail}")
print(f"  Result: {'PASS ✓' if c5_pass else 'FAIL ✗'}")
criteria_results.append((5, "MARGINAL_VALUE", c5_pass, c5_detail))
decision_report.append(f"[5] MARGINAL VALUE: {'PASS' if c5_pass else 'FAIL'} — {c5_detail}")

# ============================================================================
# CRITERION 6: NO CATASTROPHIC DEGRADATION (< 25% queries degraded vs D-21)
# ============================================================================

print("\n[6/7] NO CATASTROPHIC DEGRADATION — Query Degradation Rate")
print("-" * 80)

if "multi_layer_vs_baseline" in delta_results:
    ml_bl = delta_results["multi_layer_vs_baseline"]
    degraded_n = ml_bl["degraded"].sum()
    degraded_pct = (degraded_n / len(ml_bl) * 100) if len(ml_bl) > 0 else 0

    c6_pass = degraded_pct < 25
    c6_detail = f"{degraded_n}/{len(ml_bl)} queries degraded ({degraded_pct:.1f}%). Threshold: <25%."
    print(f"  {c6_detail}")
else:
    c6_pass = False
    c6_detail = "D-21 data unavailable."
    print(f"  {c6_detail}")

print(f"  Result: {'PASS ✓' if c6_pass else 'FAIL ✗'}")
criteria_results.append((6, "NO_CATASTROPHIC_DEGRADATION", c6_pass, c6_detail))
decision_report.append(f"[6] NO CATASTROPHIC DEGRADATION: {'PASS' if c6_pass else 'FAIL'} — {c6_detail}")

# ============================================================================
# CRITERION 7: AUTHORITY/TEMPORAL BENEFIT
# ============================================================================

print("\n[7/7] AUTHORITY/TEMPORAL BENEFIT — Specialized Queries Benefit More")
print("-" * 80)

if "multi_layer_vs_baseline" in delta_results:
    ml_bl = delta_results["multi_layer_vs_baseline"]

    type_mean_deltas = {}
    for qtype in ["SINGLE_HOP", "MULTI_HOP", "AUTHORITY", "TEMPORAL", "EXPLORATORY"]:
        type_df = ml_bl[ml_bl["query_type"] == qtype]
        if not type_df.empty:
            # Average across all 4 delta metrics
            avg_delta = type_df[["delta_p5", "delta_r10", "delta_ndcg", "delta_mrr"]].mean().mean()
            type_mean_deltas[qtype] = avg_delta
            print(f"  {qtype:15} avg_delta={avg_delta:+.4f}")

    auth_better = type_mean_deltas.get("AUTHORITY", -999) > type_mean_deltas.get("SINGLE_HOP", -999)
    temp_better = type_mean_deltas.get("TEMPORAL", -999) > type_mean_deltas.get("SINGLE_HOP", -999)

    c7_pass = auth_better or temp_better
    c7_detail = f"Authority > Factual: {auth_better}. Temporal > Factual: {temp_better}. Threshold: at least one true."
else:
    c7_pass = False
    c7_detail = "D-21 data unavailable."

print(f"\n  {c7_detail}")
print(f"  Result: {'PASS ✓' if c7_pass else 'FAIL ✗'}")
criteria_results.append((7, "AUTHORITY_TEMPORAL_BENEFIT", c7_pass, c7_detail))
decision_report.append(f"[7] AUTHORITY/TEMPORAL BENEFIT: {'PASS' if c7_pass else 'FAIL'} — {c7_detail}")

# ============================================================================
# SCORING & DECISION
# ============================================================================

print(f"\n{'=' * 90}")
print("FINAL SCORING & DECISION")
print(f"{'=' * 90}")

total_pass = sum(1 for _, _, passed, _ in criteria_results if passed)
c2_passed = criteria_results[1][2]  # Criterion 2
c4_passed = criteria_results[3][2]  # Criterion 4
mandatory_met = c2_passed and c4_passed

print(f"\n  Criteria passed: {total_pass}/7")
print(f"  Mandatory conditions (2 + 4): {'MET ✓' if mandatory_met else 'NOT MET ✗'}")
print(f"    Criterion 2 (Improvement over D-21): {'PASS' if c2_passed else 'FAIL'}")
print(f"    Criterion 4 (Statistical Significance): {'PASS' if c4_passed else 'FAIL'}")

if total_pass >= 5 and mandatory_met:
    go_nogo_decision = "GO"
    decision_msg = "GO: Multi-layer enrichment is validated. Proceed to D-24 (Layer Ablation Study)."
elif total_pass >= 3:
    go_nogo_decision = "CONDITIONAL_GO"
    decision_msg = "CONDITIONAL GO: Promising but not conclusive. Proceed to D-24 with reduced scope (4-5 layers)."
else:
    go_nogo_decision = "NO_GO"
    decision_msg = "NO-GO: Multi-layer enrichment does not justify complexity. Revert to D-22 single-layer approach."

print(f"\n  ╔{'═' * 70}╗")
print(f"  ║  DECISION: {go_nogo_decision:57} ║")
print(f"  ╚{'═' * 70}╝")
print(f"\n  {decision_msg}")

# Append to decision report
decision_report.append("")
decision_report.append("=" * 90)
decision_report.append(f"SUMMARY: {total_pass}/7 criteria passed")
decision_report.append(f"MANDATORY (criteria 2+4): {'MET' if mandatory_met else 'NOT MET'}")
decision_report.append(f"DECISION: {go_nogo_decision}")
decision_report.append(f"RATIONALE: {decision_msg}")
decision_report = "\n".join(decision_report)

print("\n✓ Decision engine complete. Report stored in decision_report variable.")


In [ ]:
#!/usr/bin/env python3
"""
D-23 Cell 21: Export Results — Final Artifact Assembly

Exports all D-23 output artifacts and prints a complete manifest.

Artifacts:
  1. d23_results.csv — per-query metrics
  2. d23_delta_vs_d21.csv — already exported in Cell 16
  3. d23_delta_vs_d22.csv — already exported in Cell 16
  4. d23_layer_token_audits.csv — re-exported from Cell 10 data
  5. d23_significance_results.csv — already exported in Cell 17
  6. d23_go_nogo_decision.txt — formal decision report
  7. Visualizations — already saved in Cells 18-19

Reference: D-22 Cell 20 (export pattern)
"""

import os

print("=" * 90)
print("EXPORTING D-23 RESULTS — FINAL ARTIFACT ASSEMBLY")
print("=" * 90)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = []

# ---- 1. D-23 Metrics Results ----
results_path = OUTPUT_DIR / "d23_results.csv"
d23_export = d23_df[["model", "experiment", "query_id", "query_type"] + METRICS].copy()
d23_export.to_csv(results_path, index=False)
size = results_path.stat().st_size
manifest.append(("d23_results.csv", size, f"{len(d23_export)} queries"))
print(f"\n✓ d23_results.csv ({size:,} bytes, {len(d23_export)} queries)")

# ---- 2-3. Delta CSVs (verify) ----
for delta_file, desc in [
    ("d23_delta_vs_d21.csv", "ML vs BL deltas"),
    ("d23_delta_vs_d22.csv", "ML vs SL deltas"),
]:
    path = OUTPUT_DIR / delta_file
    if path.exists():
        size = path.stat().st_size
        manifest.append((delta_file, size, desc))
        print(f"✓ {delta_file} ({size:,} bytes)")
    else:
        print(f"✗ {delta_file} — not found (upstream data may be missing)")

# ---- 4. Layer Token Audit ----
if layer_token_audits:
    audit_path = OUTPUT_DIR / "d23_layer_token_audits.csv"
    audit_df = pd.DataFrame(layer_token_audits)
    audit_df.to_csv(audit_path, index=False)
    size = audit_path.stat().st_size
    manifest.append(("d23_layer_token_audits.csv", size, f"{len(audit_df)} chunks"))
    print(f"✓ d23_layer_token_audits.csv ({size:,} bytes, {len(audit_df)} chunks)")
else:
    print(f"✗ d23_layer_token_audits.csv — audit data not available")

# ---- 5. Significance Results (verify) ----
sig_path = OUTPUT_DIR / "d23_significance_results.csv"
if sig_path.exists():
    size = sig_path.stat().st_size
    manifest.append(("d23_significance_results.csv", size, "12 tests"))
    print(f"✓ d23_significance_results.csv ({size:,} bytes)")
else:
    print(f"✗ d23_significance_results.csv — not found")

# ---- 6. GO/NO-GO Decision Report ----
decision_path = OUTPUT_DIR / "d23_go_nogo_decision.txt"
with open(decision_path, "w") as f:
    f.write(decision_report)
size = decision_path.stat().st_size
manifest.append(("d23_go_nogo_decision.txt", size, f"Decision: {go_nogo_decision}"))
print(f"✓ d23_go_nogo_decision.txt ({size:,} bytes)")

# ---- 7. Visualizations (verify) ----
viz_files = [
    "visualization_3way_comparison.png",
    "visualization_delta_heatmap.png",
    "visualization_layer_token_distribution.png",
]
for viz_file in viz_files:
    viz_path = OUTPUT_DIR / viz_file
    if viz_path.exists():
        size = viz_path.stat().st_size
        manifest.append((viz_file, size, "chart"))
        print(f"✓ {viz_file} ({size:,} bytes)")
    else:
        print(f"✗ {viz_file} — not generated")

# ---- ChromaDB ----
print(f"\n✓ ChromaDB collection: {collection_name}")
print(f"  Location: {CHROMADB_DIR}")

# ============================================================================
# COMPLETE MANIFEST
# ============================================================================

print(f"\n{'=' * 90}")
print("COMPLETE EXPORT MANIFEST")
print(f"{'=' * 90}")

total_size = sum(size for _, size, _ in manifest)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Total artifacts:  {len(manifest)}")
print(f"Total size:       {total_size / 1024:.1f} KB")
print()

for filename, size, description in sorted(manifest):
    print(f"  {filename:45} {size:10,} bytes  ({description})")

# ============================================================================
# FINAL STATUS
# ============================================================================

print(f"\n{'=' * 90}")
print(f"D-23 NOTEBOOK COMPLETE")
print(f"{'=' * 90}")
print(f"\n  Model:    {SELECTED_MODEL}")
print(f"  Queries:  {len(d23_df)}")
print(f"  Decision: {go_nogo_decision}")
print(f"  Report:   {decision_path}")

if go_nogo_decision == "GO":
    print(f"\n  → NEXT: D-24 (Layer Ablation Study)")
    print(f"    Identify highest-value layers; optimize token budget.")
elif go_nogo_decision == "CONDITIONAL_GO":
    print(f"\n  → NEXT: D-24 with reduced scope (4-5 layers)")
    print(f"    Focus on top-performing layers; set simplification gate.")
else:
    print(f"\n  → NEXT: Revert to D-22 single-layer enrichment.")
    print(f"    Multi-layer complexity not justified by results.")


## GO/NO-GO Decision Summary

D-23 evaluated multi-layer enrichment (8 layers: Corpus, Domain, Entity, Authority, Temporal,
Relational, Section, Content) against two baselines:

- **D-21 (Baseline)**: No enrichment — raw chunk retrieval
- **D-22 (Single-Layer)**: Simple metadata prefix enrichment

The 7-criterion evaluation framework tested whether the additional complexity of 8 layers
justifies the engineering effort and token budget cost, using Bonferroni-corrected statistical
testing (adjusted α ≈ 0.0167).

**Decision: [TO BE FILLED AFTER EXECUTION — see Cell 20 output]**

---

## Three Scenario Paths

### Scenario A: GO (≥5/7 criteria + mandatory conditions met)

**Action**: Proceed to D-24 (Layer Ablation Study).

**D-24 Scope**:
- Sequentially remove layers to measure individual contribution
- Order layers by token cost and information value
- Build "minimum viable layers" configuration (e.g., Authority + Temporal + Content)
- Test interaction effects: does layer order matter?

**C# Implementation Path**:
- Build all 8 layer enrichers with token overflow guards
- Implement per-chunk enrichment pipeline
- Add telemetry for per-layer contribution tracking

### Scenario B: CONDITIONAL GO (3-4/7 criteria met)

**Action**: Proceed to D-24 with reduced scope and simplification target.

**Simplification Target**: Reduce from 8 layers to 4-5 high-value layers.

**D-24 Scope**:
- Focus ablation on identifying top 4-5 layers
- Test aggressive removal of mid-value layers (Domain, Relational, Section)
- Gate: if top-4 configuration ≈ D-22 performance, revert to D-22

**C# Implementation Path**:
- Build streamlined enrichment with 4-5 layers only
- Defer optional layers (toggle on/off)

### Scenario C: NO-GO (<3/7 criteria met)

**Action**: Abandon multi-layer. Standardize on D-22 (single-layer enrichment).

**Next Steps**:
- Skip D-24 entirely
- Focus on D-25 (Metadata Filtering & Re-ranking) using D-22 as baseline
- Revisit multi-layer only if future models show different layer benefits

**C# Implementation Path**:
- Simple single-layer prefix enrichment (~50 lines of code)
- Allocate token budget to metadata filtering and re-ranking

---

## What We Learned

### Token Budget Insights
- Layer token distribution reveals which layers are "expensive"
- High-token layers (e.g., Relational) are candidates for D-24 ablation
- v2-moe token pressure is real; multi-layer may require layer dropout on small models

### Query Type Performance
- Authority queries: expected to benefit most from Authority and Entity layers
- Temporal queries: expected to benefit from Temporal and Relational layers
- Factual queries: expected to show baseline semantic improvement (least sensitive)

### Statistical Confidence
- Bonferroni correction at α ≈ 0.0167 is conservative
- If results are borderline, D-24 should consider Holm-Bonferroni or FDR control

### Marginal Value
- Single-layer → multi-layer transition: what percentage improvement do layers 2-8 add?
- If marginal value < 5%, complexity may not be justified

---

## C# Implementation Implications

### If GO or CONDITIONAL GO